<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Phase_3_Statistical_Analysis/STATISTICAL_ANALYSIS_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openpyxl lifelines scikit-learn seaborn scipy gspread python-ternary --quiet
!pip install imgkit --quiet
!pip install scikit-survival --quiet
!apt-get install wkhtmltopdf --quiet

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════════════════════
import os, re, glob, zipfile, warnings, math
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import chi2, gaussian_kde

from lifelines import (KaplanMeierFitter, NelsonAalenFitter,
                       AalenJohansenFitter, CoxPHFitter)
from lifelines.statistics import multivariate_logrank_test
from sklearn.metrics import cohen_kappa_score, confusion_matrix

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
np.random.seed(42)

print("✅ Libraries loaded")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Load Data
# Upload to /content/ before running:
#   • FINAL_RESULTS.csv
#   • FINAL_VALIDATED_T2D_COHORT.csv
#   • Filtered_Study_Data.zip  (optional — only needed for raw Synthea tables)
# ═══════════════════════════════════════════════════════════════════════════════
CONTENT = "/content"

# ── 1. Main results dataframe ─────────────────────────────────────────────────
results = pd.read_csv(f"{CONTENT}/FINAL_RESULTS.csv", low_memory=False)
print(f"✅ results loaded: {len(results):,} rows")

# ── 2. Cohort ─────────────────────────────────────────────────────────────────
cohort = pd.read_csv(f"{CONTENT}/FINAL_VALIDATED_T2D_COHORT.csv", low_memory=False)
for col in ["First_Diagnosis_Date_t0", "First_Treatment_Date_t1",
            "Observation_End", "DEATHDATE", "BIRTHDATE"]:
    if col in cohort.columns:
        cohort[col] = pd.to_datetime(cohort[col], errors="coerce")

cohort["reclass_date_5"]      = cohort["First_Treatment_Date_t1"] + pd.DateOffset(years=5)
cohort["reclass_date_10"]     = cohort["First_Treatment_Date_t1"] + pd.DateOffset(years=10)
cohort["eligible_reclass_5"]  = (cohort["Observation_End"] >= cohort["reclass_date_5"]).astype(int)
cohort["eligible_reclass_10"] = (cohort["Observation_End"] >= cohort["reclass_date_10"]).astype(int)

print(f"✅ cohort loaded:  {len(cohort):,} patients")
print(f"   Eligible T5:   {cohort['eligible_reclass_5'].sum()}")
print(f"   Eligible T10:  {cohort['eligible_reclass_10'].sum()}")

# ── 3. Synthea raw tables (optional) ─────────────────────────────────────────
ZIP_PATH = f"{CONTENT}/Filtered_Study_Data.zip"
SYNTHEA_TABLES = ["conditions", "observations", "medications",
                  "encounters", "procedures", "careplans", "patients"]
raw = {}

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(CONTENT)
    print("\nLoading Synthea CSVs …")
    for tbl in SYNTHEA_TABLES:
        files_found = sorted(glob.glob(f"{CONTENT}/output_*/csv/{tbl}.csv"))
        if files_found:
            raw[tbl] = pd.concat(
                [pd.read_csv(f, low_memory=False) for f in files_found],
                ignore_index=True
            )
            print(f"  ✅ {tbl}: {len(raw[tbl]):,} rows")
        else:
            print(f"  ⚠️  {tbl}: not found")
else:
    print("\nⓘ  Filtered_Study_Data.zip not found — skipping Synthea tables")

con_raw  = raw.get("conditions",   pd.DataFrame())
obs_raw  = raw.get("observations", pd.DataFrame())
med_raw  = raw.get("medications",  pd.DataFrame())
enc_raw  = raw.get("encounters",   pd.DataFrame())
proc_raw = raw.get("procedures",   pd.DataFrame())
care_raw = raw.get("careplans",    pd.DataFrame())
pat_raw  = raw.get("patients",     pd.DataFrame())

for df in [con_raw, obs_raw, med_raw, enc_raw, proc_raw, care_raw]:
    if "PATIENT" in df.columns:
        df["PATIENT"] = df["PATIENT"].astype(str).str.strip()

print("\n✅ Setup complete — ready for analysis")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Descriptive Analysis: Population Overview
# Inputs:  cohort, results, pat_raw
# Outputs: display + /content/results/descriptive_population_overview.csv
# ═══════════════════════════════════════════════════════════════════════════════
from IPython.display import display, HTML, Markdown
import os

RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Formatters ─────────────────────────────────────────────────────────────────
def n_pct(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    n, total = int(s.sum()), len(s)
    return f"{n:,} ({100*n/total:.1f}%)" if total else "—"

def pct_cat(series, val):
    s = series.astype(str).str.strip().str.lower()
    n, total = int((s == val.lower()).sum()), len(s)
    return f"{n:,} ({100*n/total:.1f}%)" if total else "—"

def med_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0: return "—"
    return f"{s.median():.1f} [{s.quantile(.25):.1f}–{s.quantile(.75):.1f}]"

# ── 1. Full cohort prep ────────────────────────────────────────────────────────
cf = cohort.copy()
cf["PATIENT"] = cf["PATIENT"].astype(str).str.strip()

cf["GENDER"] = (cf["GENDER"].astype(str).str.upper().str[0]
                .where(cf["GENDER"].astype(str).str.upper().str[0]
                       .isin(["M","F"]), np.nan))

cf["AGE_AT_DIAGNOSIS"]        = pd.to_numeric(cf["Age_at_Diagnosis"], errors="coerce")
cf["First_Treatment_Date_t1"] = pd.to_datetime(cf["First_Treatment_Date_t1"], errors="coerce")
cf["BIRTHDATE"]               = pd.to_datetime(cf["BIRTHDATE"], errors="coerce")
cf["T5_DATE"]                 = cf["First_Treatment_Date_t1"] + pd.DateOffset(years=5)
cf["AGE_AT_T5"]               = ((cf["T5_DATE"] - cf["BIRTHDATE"]).dt.days / 365.25).round(1)
cf_t5_age                     = cf.loc[cf["eligible_reclass_5"] == 1, "AGE_AT_T5"]

# Normalise drug class labels
def normalise_drug(val):
    v = str(val).lower()
    if "metformin"   in v: return "Metformin"
    if "insulin"     in v: return "Insulin"
    if "liraglutide" in v: return "Liraglutide (GLP-1)"
    if "glp"         in v: return "Liraglutide (GLP-1)"
    return str(val).strip()

cf["FIRST_DRUG_CLASS"] = cf["First_Treatment_Drug_Class"].apply(normalise_drug)

# Attach RACE from pat_raw (hispanic lives in RACE in this Synthea version;
# ETHNICITY stores ancestry like "irish" / "puerto_rican" — not used here)
if len(pat_raw) > 0:
    pat_demo = (pat_raw[["ID","RACE"]].copy()
                .rename(columns={"ID":"PATIENT"}))
    pat_demo["PATIENT"] = pat_demo["PATIENT"].astype(str).str.strip()
    pat_demo["RACE"]    = pat_demo["RACE"].astype(str).str.lower().str.strip()
    cf = cf.merge(pat_demo, on="PATIENT", how="left")
    print(f"RACE attached — top values:\n{cf['RACE'].value_counts().head(6).to_string()}")
    print(f"Missing race: {cf['RACE'].isna().sum():,} of {len(cf):,}")
else:
    print("⚠️  pat_raw empty — re-run Cell 2 with Filtered_Study_Data.zip uploaded")
    cf["RACE"] = np.nan

print(f"\nFull cohort:         {len(cf):,}")
print(f"T5-eligible:         {cf['eligible_reclass_5'].sum():,}")
print(f"Age at T5 computed:  {cf_t5_age.notna().sum():,}")
print(f"\nDrug class (normalised):\n{cf['FIRST_DRUG_CLASS'].value_counts().head(6).to_string()}")

# ── 2. Analytic cohorts ────────────────────────────────────────────────────────
_demo = cf[["PATIENT","FIRST_DRUG_CLASS","AGE_AT_DIAGNOSIS","RACE"]]
res   = results.drop(columns=["RACE","ETHNICITY"], errors="ignore").merge(_demo, on="PATIENT", how="left")

df_cw = res[res["LLM_AVAILABLE"] == True].copy()
df_ow = res[res["OW_AVAILABLE"]  == True].copy()

print(f"\nCW pipeline: {len(df_cw):,}  |  OW pipeline: {len(df_ow):,}")

# ── 3. Build table ─────────────────────────────────────────────────────────────
rows = []
def row(label, full, cw, ow, indent=False):
    rows.append({"Characteristic": ("  " if indent else "") + label,
                 "_full": full, "_cw": cw, "_ow": ow})
def section(label):
    row(label, "", "", "")

# Population
row("N", f"{len(cf):,}", f"{len(df_cw):,}", f"{len(df_ow):,}")
row("T\u2085-eligible, n (%)",
    n_pct(cf["eligible_reclass_5"]),
    "100% (by design)", "100% (by design)")

# Sex
section("Sex, n (%)")
for g, lbl in [("F","Female"), ("M","Male")]:
    row(lbl,
        pct_cat(cf["GENDER"],     g),
        pct_cat(df_cw["GENDER"],  g),
        pct_cat(df_ow["GENDER"],  g), indent=True)

# Race (all from RACE column; hispanic included there in Synthea)
section("Race, n (%)")
for val, lbl in [("white","White"), ("hispanic","Hispanic"),
                 ("black","Black"), ("asian","Asian")]:
    row(lbl,
        pct_cat(cf["RACE"],     val),
        pct_cat(df_cw["RACE"],  val),
        pct_cat(df_ow["RACE"],  val), indent=True)

# Age
row("Age at T2D diagnosis (years), median [IQR]",
    med_iqr(cf["AGE_AT_DIAGNOSIS"]),
    med_iqr(df_cw["AGE_AT_DIAGNOSIS"]),
    med_iqr(df_ow["AGE_AT_DIAGNOSIS"]))

row("Age at T\u2085 index (years), median [IQR]",
    med_iqr(cf_t5_age),
    med_iqr(df_cw["AGE_AT_T5"]),
    med_iqr(df_ow["AGE_AT_T5"]))

# First drug class
section("First pharmacological treatment, n (%)")
for drug in ["Metformin", "Insulin", "Liraglutide (GLP-1)"]:
    row(drug,
        pct_cat(cf["FIRST_DRUG_CLASS"],    drug),
        pct_cat(df_cw["FIRST_DRUG_CLASS"], drug),
        pct_cat(df_ow["FIRST_DRUG_CLASS"], drug), indent=True)

# ── 4. Assemble DataFrame ──────────────────────────────────────────────────────
desc_df = pd.DataFrame(rows)
col_full = f"Full T2D Cohort (N={len(cf):,})"
col_cw   = f"CW Analytic Cohort (N={len(df_cw):,})"
col_ow   = f"OW Analytic Cohort (N={len(df_ow):,})"
desc_df.rename(columns={"_full": col_full, "_cw": col_cw, "_ow": col_ow}, inplace=True)

# ── 5. Save ────────────────────────────────────────────────────────────────────
out_path = f"{RESULTS_DIR}/descriptive_population_overview.csv"
desc_df.to_csv(out_path, index=False)
print(f"\n💾 Saved → {out_path}")

# ── 6. Display ─────────────────────────────────────────────────────────────────
def render_table(df):
    thead = "".join(f"<th>{c}</th>" for c in df.columns)
    tbody = ""
    for _, r in df.iterrows():
        char = str(r.iloc[0])
        is_section = not char.startswith("  ") and str(r.iloc[1]) == ""
        if is_section:
            s = ' style="font-weight:600;padding-top:10px;border-top:1px solid #bbb;"'
        elif char.startswith("  "):
            s = ' style="padding-left:20px;color:#444;"'
        else:
            s = ""
        tbody += f"<tr>{''.join(f'<td{s}>{v}</td>' for v in r)}</tr>\n"
    n_missing_race = int(cf["RACE"].isna().sum())
    return f"""
    <style>
      .pop-table{{font-family:Georgia,'Times New Roman',serif;font-size:13px;
        border-collapse:collapse;width:100%;margin:14px 0;}}
      .pop-table th{{background:#f4f4f4;font-weight:600;text-align:left;
        padding:7px 12px;border-top:2px solid #222;border-bottom:1.5px solid #222;}}
      .pop-table td{{padding:4px 12px;border-bottom:0.5px solid #e4e4e4;vertical-align:top;}}
      .pop-table tr:last-child td{{border-bottom:2px solid #222;}}
    </style>
    <table class="pop-table">
      <thead><tr>{thead}</tr></thead>
      <tbody>{tbody}</tbody>
    </table>
    <p style="font-size:11px;font-family:Georgia;color:#666;margin-top:4px;">
      CW = closed-weight pipeline (GPT-4o + DeepSeek-V3; Claude 3.5 Sonnet consolidator).
      OW = open-weight pipeline (Gemma 2 27B + Qwen 2.5 14B; Llama 3.1 70B consolidator).
      T\u2085 = landmark five years after first pharmacological treatment.
      Age at T\u2085 for the full cohort reflects T\u2085-eligible patients only (N={cf['eligible_reclass_5'].sum():,}).
      Race sourced from Synthea patients table; {n_missing_race:,} of {len(cf):,} patients
      have missing race data. Percentages are of all patients in each cohort.
    </p>"""

display(Markdown("### Table 1 — Baseline Characteristics"))
display(HTML(render_table(desc_df)))
print(f"\n✅ Done — {len(desc_df)} rows")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Build encounter variables
# Inputs:  results, enc_raw
# Outputs: results updated with DAYS_*/EVENT_* columns; df_cw, df_ow refreshed
# ═══════════════════════════════════════════════════════════════════════════════

analytic_ids = set(results["PATIENT"].astype(str).str.strip())

fu_lookup = results[["PATIENT","DATE_T5","OS_TIME"]].copy()
fu_lookup["FOLLOWUP_START"] = pd.to_datetime(fu_lookup["DATE_T5"], errors="coerce")
fu_lookup["FOLLOWUP_END"]   = (fu_lookup["FOLLOWUP_START"] +
                                pd.to_timedelta(fu_lookup["OS_TIME"], unit="D"))
fu_lookup = fu_lookup.set_index("PATIENT")

enc_raw["PATIENT"] = enc_raw["PATIENT"].astype(str).str.strip()
enc_raw["DATE"]    = pd.to_datetime(enc_raw["DATE"], errors="coerce")
enc_analytic = enc_raw[enc_raw["PATIENT"].isin(analytic_ids)]

enc_fu = enc_analytic.merge(
    fu_lookup[["FOLLOWUP_START","FOLLOWUP_END"]].reset_index(),
    on="PATIENT", how="left")
enc_fu = enc_fu[
    (enc_fu["DATE"] >= enc_fu["FOLLOWUP_START"]) &
    (enc_fu["DATE"] <= enc_fu["FOLLOWUP_END"])
].copy()

ENCOUNTER_WEIGHTS = {
    "Emergency room admission": 3,
    "Emergency hospital admission for asthma": 3,
    "Obstetric emergency hospital admission": 3,
    "Admission to surgical department": 4,
    "Non-urgent orthopedic admission": 2,
    "Encounter for symptom": 2,
    "Encounter for problem": 2,
    "Consultation for treatment": 2,
    "Outpatient Encounter": 1,
    "Encounter for 'check-up'": 1,
    "Asthma follow-up": 1,
    "Patient encounter procedure": 1,
    "Death Certification": 0,
}
enc_fu["WEIGHT"] = enc_fu["DESCRIPTION"].map(ENCOUNTER_WEIGHTS).fillna(1)

ENC_GROUPS = {
    "Any_Encounter":      None,
    "Emergency":          ["Emergency room admission",
                           "Emergency hospital admission for asthma",
                           "Obstetric emergency hospital admission"],
    "Outpatient":         ["Outpatient Encounter","Encounter for 'check-up'",
                           "Encounter for symptom","Encounter for problem",
                           "Consultation for treatment",
                           "Patient encounter procedure","Asthma follow-up"],
    "Inpatient_Surgical": ["Admission to surgical department",
                           "Non-urgent orthopedic admission"],
}

def days_to_first(df, descs=None):
    sub = df[df["DESCRIPTION"].isin(descs)] if descs else df[df["WEIGHT"] > 0]
    if sub.empty: return pd.Series(dtype=float)
    first = sub.groupby("PATIENT")["DATE"].min().reset_index()
    first = first.merge(fu_lookup[["FOLLOWUP_START"]].reset_index(), on="PATIENT")
    first["DAYS"] = (first["DATE"] - first["FOLLOWUP_START"]).dt.days
    return first.set_index("PATIENT")["DAYS"]

for grp_name, descs in ENC_GROUPS.items():
    col_days = f"DAYS_{grp_name.upper()}"
    col_evt  = f"EVENT_{grp_name.upper()}"
    results[col_days] = results["PATIENT"].map(days_to_first(enc_fu, descs))
    results[col_evt]  = results[col_days].notna().astype(int)
    results[col_days] = results[col_days].fillna(results["OS_TIME"])
    print(f"  {grp_name:<25} events = {int(results[col_evt].sum())}")

# Refresh pipeline subsets
df_cw = results[results["LLM_AVAILABLE"] == True].copy()
df_ow = results[results["OW_AVAILABLE"]  == True].copy()

print(f"\n✅ Encounter variables built")
print(f"   CW cohort: {len(df_cw):,}  |  OW cohort: {len(df_ow):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Table 1: Baseline Characteristics by GT Tier (all classifiers)
# Inputs:  results, df_cw, df_ow, cohort
# Outputs: display + /content/results/table1_<classifier>.csv
# ═══════════════════════════════════════════════════════════════════════════════
from IPython.display import display, HTML, Markdown

# ── Tier normalisation ─────────────────────────────────────────────────────────
TIER_ORDER = ["Baseline", "Mild", "Moderate", "Critical"]

TIER_MAP = {
    "Baseline T2D":           "Baseline",
    "Mild Complications":     "Mild",
    "Moderate Complications": "Moderate",
    "Advanced/Critical":      "Critical",
    "Baseline":  "Baseline",
    "Mild":      "Mild",
    "Moderate":  "Moderate",
    "Critical":  "Critical",
}

TIER_COLS = ["t5_young_tier", "t5_cooper_tier", "LLM_FOUR_TIER",
             "OW_FOUR_TIER",  "t5_dissco_tier", "t5_mosaic_tier"]

def normalise_tiers(df):
    for col in TIER_COLS:
        if col in df.columns:
            df[col] = df[col].map(TIER_MAP)
    return df

results = normalise_tiers(results)
df_cw   = normalise_tiers(df_cw)
df_ow   = normalise_tiers(df_ow)

print("Tier values after normalisation:")
for col in TIER_COLS:
    if col in results.columns and results[col].notna().any():
        print(f"  {col}: {results[col].value_counts(dropna=False).to_dict()}")

# ── Ensure FIRST_DRUG_CLASS is present ────────────────────────────────────────
if "FIRST_DRUG_CLASS" not in results.columns:
    def normalise_drug(val):
        v = str(val).lower()
        if "metformin"   in v: return "Metformin"
        if "insulin"     in v: return "Insulin"
        if "liraglutide" in v or "glp" in v: return "Liraglutide (GLP-1)"
        return str(val).strip()
    _drug = (cohort[["PATIENT", "First_Treatment_Drug_Class"]]
             .rename(columns={"First_Treatment_Drug_Class": "FIRST_DRUG_CLASS"})
             .copy())
    _drug["FIRST_DRUG_CLASS"] = _drug["FIRST_DRUG_CLASS"].apply(normalise_drug)
    results = results.merge(_drug, on="PATIENT", how="left")

# Normalise GENDER to M/F
results["GENDER"] = (results["GENDER"].astype(str).str.upper().str[0]
                     .where(results["GENDER"].astype(str).str.upper().str[0]
                            .isin(["M", "F"]), np.nan))

# Refresh pipeline subsets with all columns
df_cw = results[results["LLM_AVAILABLE"] == True].copy()
df_ow = results[results["OW_AVAILABLE"]  == True].copy()

print(f"\ndf_cw: {len(df_cw):,}  |  df_ow: {len(df_ow):,}")

# ── Helper functions ───────────────────────────────────────────────────────────
def compute_smd_continuous(series, groups, ref_group):
    ref = series[groups == ref_group].dropna()
    return {g: abs(ref.mean() - series[groups==g].dropna().mean()) /
               max(np.sqrt((ref.std()**2 +
                            series[groups==g].dropna().std()**2) / 2), 1e-9)
            for g in groups.unique() if g != ref_group}

def compute_smd_binary(series, groups, ref_group):
    ref   = series[groups == ref_group].dropna()
    p_ref = ref.mean()
    return {g: abs(p_ref - series[groups==g].dropna().mean()) /
               max(np.sqrt((p_ref*(1-p_ref) +
                            series[groups==g].dropna().mean() *
                           (1-series[groups==g].dropna().mean())) / 2), 1e-9)
            for g in groups.unique() if g != ref_group}

def safe_int(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).astype(int)

def fmt_mean_sd(s):  return f"{s.mean():.1f} ± {s.std():.1f}"
def fmt_n_pct(n, t): return f"{n:,} ({100*n/t:.1f}%)" if t > 0 else "—"
def fmt_med_iqr(s):  return f"{s.median():.1f} [{s.quantile(.25):.1f}–{s.quantile(.75):.1f}]"

# ── Build Table 1 ──────────────────────────────────────────────────────────────
def build_table1(df_in, tier_col, tier_order=TIER_ORDER, ref_tier="Baseline"):
    df      = df_in.dropna(subset=[tier_col]).copy()
    df      = df[df[tier_col].isin(tier_order)]
    rows    = []
    total   = len(df)
    tier_ns = {t: (df[tier_col]==t).sum() for t in tier_order}

    def add_row(label, vals_by_tier, overall_val, smd_dict=None, indent=False):
        r = {"Characteristic": ("  " if indent else "") + label,
             "Overall": overall_val}
        for t in tier_order:
            r[t] = vals_by_tier.get(t, "—")
        max_smd = max([v for v in (smd_dict or {}).values()
                       if pd.notna(v)], default=np.nan)
        r["Max SMD"] = f"{max_smd:.3f}" if pd.notna(max_smd) else "—"
        rows.append(r)

    # N
    rows.append({"Characteristic": "N", "Overall": str(total),
                 **{t: str(tier_ns[t]) for t in tier_order}, "Max SMD": "—"})

    # Age
    if "AGE_AT_T5" in df.columns:
        add_row("Age at T5, mean ± SD",
                {t: fmt_mean_sd(df[df[tier_col]==t]["AGE_AT_T5"])
                 for t in tier_order},
                fmt_mean_sd(df["AGE_AT_T5"]),
                compute_smd_continuous(df["AGE_AT_T5"], df[tier_col], ref_tier))

    # Sex
    if "GENDER" in df.columns:
        add_row("Sex", {}, "")
        for g, lbl in [("F","Female"), ("M","Male")]:
            mask = (df["GENDER"]==g).astype(int)
            add_row(lbl,
                    {t: fmt_n_pct((df[df[tier_col]==t]["GENDER"]==g).sum(), tier_ns[t])
                     for t in tier_order},
                    fmt_n_pct((df["GENDER"]==g).sum(), total),
                    compute_smd_binary(mask, df[tier_col], ref_tier),
                    indent=True)

    # First drug class
    if "FIRST_DRUG_CLASS" in df.columns:
        add_row("First drug class", {}, "")
        for drug in ["Metformin", "Insulin", "Liraglutide (GLP-1)"]:
            mask = (df["FIRST_DRUG_CLASS"]==drug).astype(int)
            add_row(drug,
                    {t: fmt_n_pct((df[df[tier_col]==t]["FIRST_DRUG_CLASS"]==drug).sum(),
                                  tier_ns[t]) for t in tier_order},
                    fmt_n_pct(mask.sum(), total),
                    compute_smd_binary(mask, df[tier_col], ref_tier),
                    indent=True)

    # DCSI
    if "t5_dcsi" in df.columns:
        add_row("DCSI score at T5, median [IQR]",
                {t: fmt_med_iqr(df[df[tier_col]==t]["t5_dcsi"])
                 for t in tier_order},
                fmt_med_iqr(df["t5_dcsi"]),
                compute_smd_continuous(df["t5_dcsi"], df[tier_col], ref_tier))

    # Complications
    add_row("Complications at T5", {}, "")
    for comp_col, lbl in [("t5_ret","Retinopathy"), ("t5_neph","Nephropathy"),
                           ("t5_neu","Neuropathy"),  ("t5_cvd","CVD"),
                           ("t5_pvd","PVD"),          ("t5_cbv","Cerebrovascular"),
                           ("t5_met","Metabolic")]:
        if comp_col not in df.columns: continue
        mask = safe_int(df[comp_col])
        add_row(lbl,
                {t: fmt_n_pct(safe_int(df[df[tier_col]==t][comp_col]).sum(),
                               tier_ns[t]) for t in tier_order},
                fmt_n_pct(mask.sum(), total),
                compute_smd_binary(mask, df[tier_col], ref_tier),
                indent=True)

    # Outcomes
    if "DIED_IN_FOLLOWUP" in df.columns:
        mask = safe_int(df["DIED_IN_FOLLOWUP"])
        add_row("Death in follow-up, n (%)",
                {t: fmt_n_pct(safe_int(df[df[tier_col]==t]["DIED_IN_FOLLOWUP"]).sum(),
                               tier_ns[t]) for t in tier_order},
                fmt_n_pct(mask.sum(), total),
                compute_smd_binary(mask, df[tier_col], ref_tier))

    if "COMP_EVENT" in df.columns:
        mask = safe_int(df["COMP_EVENT"])
        add_row("New complication in follow-up, n (%)",
                {t: fmt_n_pct(safe_int(df[df[tier_col]==t]["COMP_EVENT"]).sum(),
                               tier_ns[t]) for t in tier_order},
                fmt_n_pct(mask.sum(), total),
                compute_smd_binary(mask, df[tier_col], ref_tier))

    return pd.DataFrame(rows)

# ── HTML renderer ──────────────────────────────────────────────────────────────
def render_t1(df, title):
    thead = "".join(f"<th>{c}</th>" for c in df.columns)
    tbody = ""
    for _, r in df.iterrows():
        char = str(r.iloc[0])
        is_section = not char.startswith("  ") and str(r.iloc[1]) == ""
        if is_section:
            s = ' style="font-weight:600;padding-top:10px;border-top:1px solid #bbb;"'
        elif char.startswith("  "):
            s = ' style="padding-left:20px;color:#444;"'
        else:
            s = ""
        tbody += f"<tr>{''.join(f'<td{s}>{v}</td>' for v in r)}</tr>\n"
    return f"""
    <style>
      .t1{{font-family:Georgia,serif;font-size:12px;border-collapse:collapse;
           width:100%;margin:10px 0 24px 0;}}
      .t1 th{{background:#f4f4f4;font-weight:600;text-align:left;
              padding:6px 10px;border-top:2px solid #222;
              border-bottom:1.5px solid #222;}}
      .t1 td{{padding:3px 10px;border-bottom:0.5px solid #e4e4e4;
              vertical-align:top;}}
      .t1 tr:last-child td{{border-bottom:2px solid #222;}}
    </style>
    <h4 style="font-family:Georgia;margin-bottom:4px;">{title}</h4>
    <table class="t1"><thead><tr>{thead}</tr></thead>
    <tbody>{tbody}</tbody></table>"""

# ── Run all classifiers ────────────────────────────────────────────────────────
CLASSIFIERS = [
    (df_cw, "t5_young_tier",  "Young_GT_CW"),
    (df_cw, "t5_cooper_tier", "Cooper_GT_CW"),
    (df_cw, "LLM_FOUR_TIER",  "CW_LLM"),
    (df_cw, "OW_FOUR_TIER",   "OW_LLM_CW"),
    (df_ow, "t5_young_tier",  "Young_GT_OW"),
    (df_ow, "t5_cooper_tier", "Cooper_GT_OW"),
    (df_ow, "OW_FOUR_TIER",   "OW_LLM"),
    (df_ow, "t5_dissco_tier", "DiSSCo_OW"),
]

print("\nBuilding Table 1 for all classifiers …\n")
for df_in, tier_col, label in CLASSIFIERS:
    if tier_col not in df_in.columns or df_in[tier_col].isna().all():
        print(f"  ⚠️  Skipping {label} — {tier_col} not available")
        continue

    t1 = build_table1(df_in, tier_col)

    out_path = f"{RESULTS_DIR}/table1_{label.lower()}.csv"
    t1.to_csv(out_path, index=False)

    cohort_tag = f"CW (N={len(df_cw):,})" if df_in is df_cw else f"OW (N={len(df_ow):,})"
    display(HTML(render_t1(t1, f"Table 1 — {label}  [{cohort_tag}]")))
    print(f"  ✅ {label} ({len(t1)} rows) → {out_path}")

print(f"\n✅ All Table 1s complete — saved to {RESULTS_DIR}/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Table 2: Incidence Rates per 100 Person-Years (all classifiers)
# Inputs:  results, df_cw, df_ow
# Outputs: display + /content/results/table2_<classifier>.csv
# ═══════════════════════════════════════════════════════════════════════════════
from IPython.display import display, HTML, Markdown

# ── Incidence rate with Poisson CI ────────────────────────────────────────────
def poisson_ci(n_events, person_years, alpha=0.05):
    rate = (n_events / person_years) * 100 if person_years > 0 else np.nan
    if n_events == 0:
        lower = 0.0
        upper = (-np.log(alpha) / person_years) * 100 if person_years > 0 else np.nan
    else:
        lower = (chi2.ppf(alpha / 2,     2 * n_events)     / 2 / person_years) * 100
        upper = (chi2.ppf(1 - alpha / 2, 2 * n_events + 2) / 2 / person_years) * 100
    return rate, lower, upper

# ── Build Table 2 ─────────────────────────────────────────────────────────────
def build_table2(df_in, tier_col, tier_order=TIER_ORDER):
    df   = df_in.dropna(subset=[tier_col]).copy()
    rows = []

    def process_group(label, subset):
        n_total = len(subset)

        # All-cause mortality
        if "OS_TIME" in subset.columns and "OS_EVENT" in subset.columns:
            py       = subset["OS_TIME"].sum() / 365.25
            n_events = int(subset["OS_EVENT"].sum())
            n_cens   = int((subset["OS_EVENT"] == 0).sum())
            ir, lo, hi = poisson_ci(n_events, py)
            rows.append({
                "Severity":           label,
                "Outcome":            "All-cause mortality",
                "N":                  n_total,
                "FU_Person_Years":    round(py, 1),
                "N_events":           n_events,
                "N_censored":         n_cens,
                "N_competing_deaths": "—",
                "IR_per_100PY":       f"{ir:.2f}" if not np.isnan(ir) else "—",
                "CI_95":              f"[{lo:.2f}–{hi:.2f}]" if not np.isnan(ir) else "—",
            })

        # New diabetes complication (death = competing event)
        if "COMP_TIME_DAYS" in subset.columns and "COMP_EVENT" in subset.columns:
            py     = subset["COMP_TIME_DAYS"].sum() / 365.25
            n_comp = int((subset["COMP_EVENT"] == 1).sum())
            if "OS_EVENT" in subset.columns:
                n_competing = int(
                    ((subset["COMP_EVENT"] == 0) & (subset["OS_EVENT"] == 1)).sum()
                )
                n_cens = int(
                    ((subset["COMP_EVENT"] == 0) & (subset["OS_EVENT"] == 0)).sum()
                )
            else:
                n_competing = "unknown"
                n_cens      = int((subset["COMP_EVENT"] == 0).sum())
            ir, lo, hi = poisson_ci(n_comp, py)
            rows.append({
                "Severity":           label,
                "Outcome":            "New diabetes complication",
                "N":                  n_total,
                "FU_Person_Years":    round(py, 1),
                "N_events":           n_comp,
                "N_censored":         n_cens,
                "N_competing_deaths": n_competing,
                "IR_per_100PY":       f"{ir:.2f}" if not np.isnan(ir) else "—",
                "CI_95":              f"[{lo:.2f}–{hi:.2f}]" if not np.isnan(ir) else "—",
            })

    process_group("Overall", df)
    for tier in tier_order:
        process_group(tier, df[df[tier_col] == tier])

    return pd.DataFrame(rows)

# ── HTML renderer ──────────────────────────────────────────────────────────────
def render_t2(df, title):
    thead = "".join(f"<th>{c}</th>" for c in df.columns)
    tbody = ""
    for _, r in df.iterrows():
        is_overall = str(r.iloc[0]) == "Overall"
        s = ' style="font-weight:600;border-top:1px solid #bbb;"' if is_overall else ""
        tbody += f"<tr>{''.join(f'<td{s}>{v}</td>' for v in r)}</tr>\n"
    return f"""
    <style>
      .t2{{font-family:Georgia,serif;font-size:12px;border-collapse:collapse;
           width:100%;margin:10px 0 24px 0;}}
      .t2 th{{background:#f4f4f4;font-weight:600;text-align:left;
              padding:6px 10px;border-top:2px solid #222;
              border-bottom:1.5px solid #222;}}
      .t2 td{{padding:3px 10px;border-bottom:0.5px solid #e4e4e4;
              vertical-align:top;}}
      .t2 tr:last-child td{{border-bottom:2px solid #222;}}
    </style>
    <h4 style="font-family:Georgia;margin-bottom:4px;">{title}</h4>
    <table class="t2"><thead><tr>{thead}</tr></thead>
    <tbody>{tbody}</tbody></table>"""

# ── Run all classifiers ────────────────────────────────────────────────────────
print("Building Table 2 for all classifiers …\n")
for df_in, tier_col, label in CLASSIFIERS:
    if tier_col not in df_in.columns or df_in[tier_col].isna().all():
        print(f"  ⚠️  Skipping {label} — {tier_col} not available")
        continue

    t2 = build_table2(df_in, tier_col)
    t2.insert(0, "classifier", label)

    out_path = f"{RESULTS_DIR}/table2_{label.lower()}.csv"
    t2.to_csv(out_path, index=False)

    cohort_tag = f"CW (N={len(df_cw):,})" if df_in is df_cw else f"OW (N={len(df_ow):,})"
    display(HTML(render_t2(t2, f"Table 2 — {label}  [{cohort_tag}]")))
    print(f"  ✅ {label} ({len(t2)} rows) → {out_path}")

print(f"\n✅ All Table 2s complete — saved to {RESULTS_DIR}/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Survival Curve Coordinates + Cox HR Tables
# Inputs:  results, df_cw, df_ow
# Outputs: /content/results/survival_curves_*.csv, cox_hr_tables.csv
# ═══════════════════════════════════════════════════════════════════════════════
n_cw = len(df_cw)
n_ow = len(df_ow)
print(f"Pre-computing survival curve coordinates … (CW: {n_cw:,}, OW: {n_ow:,})")

km_rows, na_rows, cic_mort_rows, cic_comp_rows, cox_rows, enc_curve_rows = [], [], [], [], [], []

# ── Curve helpers ──────────────────────────────────────────────────────────────
def _km_coords(sub, time_col, event_col):
    if len(sub) == 0 or int(sub[event_col].sum()) < 2:
        return [0, 1825], [1.0, 1.0], [], [], True
    kmf = KaplanMeierFitter()
    kmf.fit(sub[time_col], sub[event_col])
    t     = kmf.survival_function_.index.tolist()
    sf    = kmf.survival_function_.iloc[:, 0].tolist()
    ci_lo = kmf.confidence_interval_.iloc[:, 0].tolist()
    ci_hi = kmf.confidence_interval_.iloc[:, 1].tolist()
    return t, sf, ci_lo, ci_hi, False

def _na_coords(sub, time_col, event_col):
    if len(sub) == 0 or int(sub[event_col].sum()) < 2:
        return [0, 1825], [0.0, 0.0], [], [], True
    naf = NelsonAalenFitter()
    naf.fit(sub[time_col], sub[event_col])
    t     = naf.cumulative_hazard_.index.tolist()
    ch    = naf.cumulative_hazard_.iloc[:, 0].tolist()
    ci_lo = naf.confidence_interval_.iloc[:, 0].tolist()
    ci_hi = naf.confidence_interval_.iloc[:, 1].tolist()
    return t, ch, ci_lo, ci_hi, False

def _cic_coords(sub, time_col, event_col, competing_col):
    if len(sub) == 0:
        return [0, 1825], [0.0, 0.0], [0.0, 0.0], True
    event_type = np.zeros(len(sub), dtype=int)
    event_type[sub[event_col].values == 1] = 1
    if competing_col and competing_col in sub.columns:
        event_type[(sub[competing_col].values == 1) & (sub[event_col].values == 0)] = 2
    if int((event_type == 1).sum()) < 2:
        return [0, 1825], [0.0, 0.0], [0.0, 0.0], True
    ajf = AalenJohansenFitter(calculate_variance=True)
    ajf.fit(durations=sub[time_col].values, event_observed=event_type, event_of_interest=1)
    t   = ajf.cumulative_density_.index.tolist()
    cif = ajf.cumulative_density_["CIF_1"].tolist()
    std = np.sqrt(ajf.variance_.clip(0)).tolist()
    return t, cif, std, False

def _base_row(analysis, tier, n, events, flat):
    return {"analysis": analysis, "tier": tier, "n": n, "events": events, "flat": flat}

# ── All analyses ───────────────────────────────────────────────────────────────
ALL_ANALYSES = [
    ("CW_Young_GT",  df_cw, "t5_young_tier"),
    ("CW_Cooper_GT", df_cw, "t5_cooper_tier"),
    ("CW_DiSSCo_GT", df_cw, "t5_dissco_tier"),
    ("CW_MOSAIC",    df_cw, "t5_mosaic_tier"),
    ("CW_LLM",       df_cw, "LLM_FOUR_TIER"),
    ("CW_OW_LLM",    df_cw, "OW_FOUR_TIER"),
    ("OW_Young_GT",  df_ow, "t5_young_tier"),
    ("OW_Cooper_GT", df_ow, "t5_cooper_tier"),
    ("OW_DiSSCo_GT", df_ow, "t5_dissco_tier"),
    ("OW_MOSAIC",    df_ow, "t5_mosaic_tier"),
    ("OW_LLM",       df_ow, "OW_FOUR_TIER"),
]

for analysis, data, tier_col in ALL_ANALYSES:
    if tier_col not in data.columns:
        print(f"  ⚠️  Skipping {analysis} — {tier_col} not available")
        continue
    df_clean = data.dropna(subset=[tier_col]).copy()

    for tier in TIER_ORDER:
        sub = df_clean[df_clean[tier_col] == tier]
        if len(sub) == 0:
            continue

        # KM mortality
        if {"OS_TIME","OS_EVENT"}.issubset(sub.columns):
            try:
                t, sf, ci_lo, ci_hi, flat = _km_coords(sub, "OS_TIME", "OS_EVENT")
                km_rows.append({**_base_row(analysis, tier, len(sub),
                                            int(sub["OS_EVENT"].sum()), flat),
                                 "outcome":    "mortality",
                                 "x_json":     json.dumps([round(v,1) for v in t]),
                                 "y_json":     json.dumps([round(v,6) for v in sf]),
                                 "ci_lo_json": json.dumps([round(v,6) for v in ci_lo]),
                                 "ci_hi_json": json.dumps([round(v,6) for v in ci_hi])})
            except Exception as e: print(f"  ⚠️ KM {analysis}/{tier}: {e}")

        # Nelson-Aalen mortality
        if {"OS_TIME","OS_EVENT"}.issubset(sub.columns):
            try:
                t, ch, ci_lo, ci_hi, flat = _na_coords(sub, "OS_TIME", "OS_EVENT")
                na_rows.append({**_base_row(analysis, tier, len(sub),
                                            int(sub["OS_EVENT"].sum()), flat),
                                 "outcome":    "mortality",
                                 "x_json":     json.dumps([round(v,1) for v in t]),
                                 "y_json":     json.dumps([round(v,6) for v in ch]),
                                 "ci_lo_json": json.dumps([round(v,6) for v in ci_lo]),
                                 "ci_hi_json": json.dumps([round(v,6) for v in ci_hi])})
            except Exception as e: print(f"  ⚠️ NA {analysis}/{tier}: {e}")

        # CIC mortality
        if {"OS_TIME","OS_EVENT"}.issubset(sub.columns):
            try:
                t, cif, std, flat = _cic_coords(sub, "OS_TIME", "OS_EVENT", None)
                cic_mort_rows.append({**_base_row(analysis, tier, len(sub),
                                                   int((sub["OS_EVENT"]==1).sum()), flat),
                                       "outcome": "mortality",
                                       "x_json":  json.dumps([round(v,1) for v in t]),
                                       "y_json":  json.dumps([round(v,6) for v in cif])})
            except Exception as e: print(f"  ⚠️ CIC-mort {analysis}/{tier}: {e}")

        # CIC complication
        if {"COMP_TIME_DAYS","COMP_EVENT"}.issubset(sub.columns):
            try:
                t, cif, std, flat = _cic_coords(sub, "COMP_TIME_DAYS", "COMP_EVENT", "OS_EVENT")
                cic_comp_rows.append({**_base_row(analysis, tier, len(sub),
                                                   int((sub["COMP_EVENT"]==1).sum()), flat),
                                       "outcome": "complication",
                                       "x_json":  json.dumps([round(v,1) for v in t]),
                                       "y_json":  json.dumps([round(v,6) for v in cif])})
            except Exception as e: print(f"  ⚠️ CIC-comp {analysis}/{tier}: {e}")

    # Cox adjusted curves
    if {"OS_TIME","AGE_AT_T5"}.issubset(df_clean.columns):
        try:
            df_cox = df_clean.dropna(subset=[tier_col,"AGE_AT_T5","GENDER",
                                             "OS_TIME","OS_EVENT"]).copy()
            df_cox["MALE"] = (df_cox["GENDER"]=="M").astype(int)
            mean_age       = df_cox["AGE_AT_T5"].mean()
            mean_male      = df_cox["MALE"].mean()
            usable         = [t for t in TIER_ORDER
                              if t != "Baseline" and t in df_cox[tier_col].unique()]
            for t in usable:
                df_cox[t] = (df_cox[tier_col]==t).astype(float)
            cph      = CoxPHFitter(penalizer=0.01)
            fit_cols = ["OS_TIME","OS_EVENT","AGE_AT_T5","MALE"] + usable
            cph.fit(df_cox[fit_cols], duration_col="OS_TIME", event_col="OS_EVENT")
            for tier in TIER_ORDER:
                sub = df_cox[df_cox[tier_col]==tier]
                if len(sub) == 0: continue
                profile = {"AGE_AT_T5": mean_age, "MALE": mean_male,
                           **{t: 1.0 if t==tier else 0.0 for t in usable}}
                sf    = cph.predict_survival_function(pd.DataFrame([profile]))
                t_arr = sf.index.tolist()
                s_arr = sf.iloc[:,0].tolist()
                cox_rows.append({**_base_row(analysis, tier, len(sub),
                                             int(sub["OS_EVENT"].sum()), False),
                                  "mean_age":      round(mean_age, 1),
                                  "mean_male_pct": round(mean_male, 3),
                                  "x_json":        json.dumps([round(v,1) for v in t_arr]),
                                  "y_json":        json.dumps([round(v,6) for v in s_arr])})
        except Exception as e: print(f"  ⚠️ Cox {analysis}: {e}")

# Encounter CIC curves (OW cohort — largest N)
for analysis, data, tier_col in [
    ("OW_Young_GT",  df_ow, "t5_young_tier"),
    ("OW_Cooper_GT", df_ow, "t5_cooper_tier"),
    ("OW_DiSSCo_GT", df_ow, "t5_dissco_tier"),
    ("OW_MOSAIC",    df_ow, "t5_mosaic_tier"),
    ("OW_LLM",       df_ow, "OW_FOUR_TIER"),
]:
    if tier_col not in data.columns: continue
    df_clean = data.dropna(subset=[tier_col]).copy()
    for grp_name in ENC_GROUPS:
        tcol = f"DAYS_{grp_name.upper()}"
        ecol = f"EVENT_{grp_name.upper()}"
        if tcol not in df_clean.columns: continue
        for tier in TIER_ORDER:
            sub = df_clean[df_clean[tier_col]==tier]
            if len(sub) == 0: continue
            try:
                t, cif, std, flat = _cic_coords(sub, tcol, ecol, "OS_EVENT")
                enc_curve_rows.append({
                    "analysis":       analysis, "encounter_type": grp_name,
                    "tier":           tier,     "n":              len(sub),
                    "events":         int((sub[ecol]==1).sum()), "flat": flat,
                    "x_json":         json.dumps([round(v,1) for v in t]),
                    "y_json":         json.dumps([round(v,6) for v in cif])})
            except Exception as e: print(f"  ⚠️ EncCIC {analysis}/{grp_name}/{tier}: {e}")

# ── Save all curve data ────────────────────────────────────────────────────────
for name, rows in [("survival_curves_km",       km_rows),
                   ("survival_curves_na",        na_rows),
                   ("survival_curves_cic_mort",  cic_mort_rows),
                   ("survival_curves_cic_comp",  cic_comp_rows),
                   ("cox_adjusted_curves",       cox_rows),
                   ("survival_curves_enc",       enc_curve_rows)]:
    if rows:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(f"{RESULTS_DIR}/{name}.csv", index=False)
        print(f"  💾 {name}: {len(df_out):,} rows")

# ── Cox HR tables ──────────────────────────────────────────────────────────────
print("\nComputing Cox HR tables …")
hr_rows = []
for analysis, data, tier_col in ALL_ANALYSES:
    if tier_col not in data.columns: continue
    for adj_label, covariates in [("crude",[]), ("adjusted",["AGE_AT_T5","MALE"])]:
        try:
            df_cox = data.dropna(subset=[tier_col,"OS_TIME","OS_EVENT"]).copy()
            if "MALE" not in df_cox.columns and "GENDER" in df_cox.columns:
                df_cox["MALE"] = (df_cox["GENDER"]=="M").astype(int)
            usable = [t for t in TIER_ORDER
                      if t != "Baseline" and t in df_cox[tier_col].unique()]
            for t in usable:
                df_cox[t] = (df_cox[tier_col]==t).astype(float)
            fit_cols = (["OS_TIME","OS_EVENT"] +
                        [c for c in covariates if c in df_cox.columns] + usable)
            cph = CoxPHFitter(penalizer=0.01)
            cph.fit(df_cox[fit_cols], duration_col="OS_TIME", event_col="OS_EVENT")
            for tier in usable:
                r = cph.summary.loc[tier]
                hr_rows.append({"analysis": analysis, "tier_col": tier_col,
                                 "adjustment": adj_label, "tier": tier,
                                 "HR":      round(math.exp(r["coef"]), 3),
                                 "CI_lo":   round(math.exp(r["coef lower 95%"]), 3),
                                 "CI_hi":   round(math.exp(r["coef upper 95%"]), 3),
                                 "p_value": round(r["p"], 4)})
        except Exception as e: print(f"  ⚠️ HR {analysis}/{adj_label}: {e}")

if hr_rows:
    hr_df = pd.DataFrame(hr_rows)
    hr_df.to_csv(f"{RESULTS_DIR}/cox_hr_tables.csv", index=False)
    display(hr_df)
    print(f"  💾 cox_hr_tables: {len(hr_df):,} rows")

print("\n✅ Survival curves + Cox HRs complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Dirichlet-Multinomial MCMC
# Inputs:  df_cw, df_ow, TIER_ORDER
# Outputs: idatas_ow, idatas_cw  (used by Cells 9 & 10)
# ═══════════════════════════════════════════════════════════════════════════════
import pymc as pm
import arviz as az
import logging
logging.getLogger("pytensor").setLevel(logging.CRITICAL)
import pytensor
pytensor.config.blas__ldflags = ""

K = len(TIER_ORDER)

PALETTE = {
    "Young GT":      "#4daf4a",
    "Cooper GT":     "#377eb8",
    "DiSSCo GT":     "#a65628",
    "MOSAIC Frozen": "#d4a017",
    "Open-Weight":   "#e41a1c",
    "Closed-Weight": "#984ea3",
}

MCMC_KWARGS = dict(draws=2000, tune=1000, chains=4,
                   target_accept=0.9, random_seed=42,
                   progressbar=True, return_inferencedata=True)

def get_counts(df, col):
    df_v   = df.dropna(subset=[col])
    counts = (df_v[col].value_counts()
              .reindex(TIER_ORDER).fillna(0)
              .values.astype(int))
    return counts, len(df_v)

def fit_dirichlet(counts, name):
    n_total = int(counts.sum())
    print(f"  Fitting: {name}  (n={n_total}, counts={counts})")
    with pm.Model():
        frac  = pm.Dirichlet("frac", a=np.ones(K), shape=K)
        _     = pm.Multinomial("obs", n=n_total, p=frac, observed=counts)
        idata = pm.sample(**MCMC_KWARGS)
    return idata

def convergence_ok(idata, name):
    rhat = az.rhat(idata)["frac"].values
    ess  = az.ess(idata)["frac"].values
    ok   = all(rhat < 1.01) and all(ess > 400)
    print(f"    {name}: R̂ max={rhat.max():.4f}  ESS min={ess.min():.0f}  "
          f"{'✅' if ok else '⚠️ CHECK'}")
    return ok

# ── OW cohort ─────────────────────────────────────────────────────────────────
print("="*65)
print(f"OW cohort  N={n_ow:,}  — fitting models")
print("="*65)

OW_CLASSIFIERS = [("Young GT",      "t5_young_tier"),
                  ("Cooper GT",     "t5_cooper_tier"),
                  ("DiSSCo GT",     "t5_dissco_tier"),
                  ("MOSAIC Frozen", "t5_mosaic_tier"),
                  ("Open-Weight",   "OW_FOUR_TIER")]

idatas_ow, counts_ow = {}, {}
for name, col in OW_CLASSIFIERS:
    if col not in df_ow.columns or df_ow[col].isna().all():
        print(f"  ⚠️  {col} not found — skipping {name}"); continue
    counts, n = get_counts(df_ow, col)
    counts_ow[name] = (counts, n)
    idatas_ow[name] = fit_dirichlet(counts, name)

print("\nConvergence — OW cohort:")
for name in idatas_ow: convergence_ok(idatas_ow[name], name)

# ── CW cohort ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print(f"CW cohort  N={n_cw:,}  — fitting models")
print("="*65)

CW_CLASSIFIERS = [("Young GT",      "t5_young_tier"),
                  ("Cooper GT",     "t5_cooper_tier"),
                  ("DiSSCo GT",     "t5_dissco_tier"),
                  ("MOSAIC Frozen", "t5_mosaic_tier"),
                  ("Open-Weight",   "OW_FOUR_TIER"),
                  ("Closed-Weight", "LLM_FOUR_TIER")]

idatas_cw, counts_cw = {}, {}
for name, col in CW_CLASSIFIERS:
    if col not in df_cw.columns or df_cw[col].isna().all():
        print(f"  ⚠️  {col} not found — skipping {name}"); continue
    counts, n = get_counts(df_cw, col)
    counts_cw[name] = (counts, n)
    idatas_cw[name] = fit_dirichlet(counts, name)

print("\nConvergence — CW cohort:")
for name in idatas_cw: convergence_ok(idatas_cw[name], name)

print(f"\n✅ Dirichlet models fitted — OW: {len(idatas_ow)}, CW: {len(idatas_cw)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 — KDE Posterior Plots + Bayesian Bias Tests
# Inputs:  idatas_ow, idatas_cw, TIER_ORDER, PALETTE
# Outputs: display + /content/results/dirichlet_kde_*.png/pdf
#                  + /content/results/dirichlet_bias_tests.csv
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import gaussian_kde

def _save_fig(fig, stem):
    for ext in ["pdf","png"]:
        fig.savefig(f"{RESULTS_DIR}/{stem}.{ext}", bbox_inches="tight", dpi=300)
    print(f"  💾 {stem}.pdf / .png")

# ── KDE panel helper ───────────────────────────────────────────────────────────
def _draw_kde_panel(ax, idatas, names, tier_idx, tier, cohort_label):
    for name in names:
        if name not in idatas: continue
        samples  = idatas[name].posterior["frac"].values.reshape(-1, K)[:, tier_idx]
        mean     = float(samples.mean())
        hdi_vals = az.hdi(samples, hdi_prob=0.95)
        lo, hi   = float(hdi_vals[0]), float(hdi_vals[1])
        color    = PALETTE[name]
        kde      = gaussian_kde(samples, bw_method=0.12)
        x        = np.linspace(0, 1, 500)
        ax.plot(x, kde(x), color=color, lw=1.8, label=name, alpha=0.9)
        ax.fill_between(np.linspace(lo, hi, 300),
                        kde(np.linspace(lo, hi, 300)),
                        alpha=0.12, color=color)
        ax.axvline(mean, color=color, lw=1.0, linestyle="--", alpha=0.65)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Tier probability  π", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title(f"{tier}  ·  {cohort_label}",
                 fontsize=9.5, fontweight="bold", loc="left", pad=4)

def _add_legend_top(fig, names, ncol=None):
    handles = [mpatches.Patch(color=PALETTE[n], label=n)
               for n in names if n in PALETTE]
    fig.legend(handles=handles, loc="upper center",
               ncol=ncol or len(handles), fontsize=9,
               frameon=False, bbox_to_anchor=(0.5, 1.01))

# ── Plot 1: OW | CW side by side ──────────────────────────────────────────────
def plot_posterior_kdes():
    fig, axes = plt.subplots(K, 2, figsize=(13, 11),
                              gridspec_kw={"hspace":0.6,"wspace":0.35})
    fig.suptitle(
        "Posterior Tier Probabilities — Bayesian Dirichlet-Multinomial\n"
        "Each curve = posterior KDE · Dashed = posterior mean · Shaded = 95% HDI",
        fontsize=12, fontweight="bold", y=1.05)
    for idatas, names, label, col_idx in [
        (idatas_ow, list(idatas_ow.keys()), f"OW Cohort  (N={n_ow:,})", 0),
        (idatas_cw, list(idatas_cw.keys()), f"CW Cohort  (N={n_cw:,})", 1),
    ]:
        for tier_idx, tier in enumerate(TIER_ORDER):
            _draw_kde_panel(axes[tier_idx, col_idx], idatas, names, tier_idx, tier, label)
    all_names = list(idatas_ow.keys()) + [n for n in idatas_cw if n not in idatas_ow]
    _add_legend_top(fig, all_names)
    plt.tight_layout(rect=[0,0,1,0.98])
    _save_fig(fig, "dirichlet_kde_plot1")
    plt.show()

# ── Plot 2: Focused comparisons ────────────────────────────────────────────────
def plot_posterior_kdes_focused():
    LEFT_NAMES  = ["Open-Weight", "MOSAIC Frozen"]
    RIGHT_NAMES = ["Open-Weight", "Cooper GT", "Young GT"]
    fig, axes = plt.subplots(K, 2, figsize=(13, 11),
                              gridspec_kw={"hspace":0.6,"wspace":0.35})
    fig.suptitle(
        "Posterior Tier Probabilities — Focused Comparison\n"
        "Each curve = posterior KDE · Dashed = posterior mean · Shaded = 95% HDI",
        fontsize=12, fontweight="bold", y=1.05)
    for idatas, names, label, col_idx in [
        (idatas_ow, LEFT_NAMES,  f"OW vs MOSAIC Frozen  (N={n_ow:,})", 0),
        (idatas_ow, RIGHT_NAMES, f"OW vs Ground Truths  (N={n_ow:,})", 1),
    ]:
        for tier_idx, tier in enumerate(TIER_ORDER):
            _draw_kde_panel(axes[tier_idx, col_idx], idatas, names, tier_idx, tier, label)
    _add_legend_top(fig, LEFT_NAMES + [n for n in RIGHT_NAMES if n not in LEFT_NAMES])
    plt.tight_layout(rect=[0,0,1,0.98])
    _save_fig(fig, "dirichlet_kde_plot2_focused")
    plt.show()

plot_posterior_kdes()
plot_posterior_kdes_focused()

# ── Bayesian bias tests ────────────────────────────────────────────────────────
def format_hdi(samples, is_delta=False):
    mean    = float(samples.mean())
    hdi_val = az.hdi(samples, hdi_prob=0.95)
    lo, hi  = float(hdi_val[0]), float(hdi_val[1])
    return (f"{mean:+.3f} [{lo:+.3f}, {hi:+.3f}]" if is_delta
            else f"{mean:.3f} [{lo:.3f}, {hi:.3f}]")

def check_bias(samples):
    hdi_val = az.hdi(samples, hdi_prob=0.95)
    lo, hi  = float(hdi_val[0]), float(hdi_val[1])
    if lo > 0: return "↑ Yes"
    if hi < 0: return "↓ Yes"
    return "No"

print("\nComputing Bayesian Δk bias tests relative to Open-Weight baseline …\n")
n_ow_samp = idatas_ow["Open-Weight"].posterior["frac"].values.reshape(-1, K).shape[0]
n_cw_samp = idatas_cw["Closed-Weight"].posterior["frac"].values.reshape(-1, K).shape[0]
n_align   = min(n_ow_samp, n_cw_samp)
rng       = np.random.default_rng(42)
idx_ow    = rng.choice(n_ow_samp, n_align, replace=False)
idx_cw    = rng.choice(n_cw_samp, n_align, replace=False)

bias_rows = []
for tier_idx, tier in enumerate(TIER_ORDER):
    def _ow(name): return idatas_ow[name].posterior["frac"].values.reshape(-1,K)[idx_ow, tier_idx]
    def _cw(name): return idatas_cw[name].posterior["frac"].values.reshape(-1,K)[idx_cw, tier_idx]
    ow_samp = _ow("Open-Weight")
    row = {"Severity Tier": tier, "Open-Weight (base)": format_hdi(ow_samp)}
    for name in ["Young GT","Cooper GT","DiSSCo GT","MOSAIC Frozen"]:
        if name not in idatas_ow: continue
        samp  = _ow(name); delta = samp - ow_samp
        row[name]             = format_hdi(samp)
        row[f"Δk {name}"]     = format_hdi(delta, is_delta=True)
        row[f"Bias? {name}"]  = check_bias(delta)
    if "Closed-Weight" in idatas_cw:
        samp  = _cw("Closed-Weight"); delta = samp - ow_samp
        row["Closed-Weight †"]     = format_hdi(samp)
        row["Δk Closed †"]         = format_hdi(delta, is_delta=True)
        row["Bias? Closed †"]      = check_bias(delta)
    bias_rows.append(row)

bias_df = pd.DataFrame(bias_rows)
print(f"Δk = classifier − Open-Weight  |  95% HDI")
print(f"† Closed-Weight: CW cohort (N={n_cw:,}); others: OW cohort (N={n_ow:,}).")
print(f"  Samples aligned to {n_align:,} for comparability.\n")
display(bias_df)
bias_df.to_csv(f"{RESULTS_DIR}/dirichlet_bias_tests.csv", index=False)
print(f"\n💾 Saved → {RESULTS_DIR}/dirichlet_bias_tests.csv")
print("✅ KDE plots + bias tests complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10 — Cohen's κ (classifier vs GT + inter-assessor)
# Inputs:  results, df_cw, df_ow
# Outputs: display + kappa_table.csv, confusion_matrices.csv
# ═══════════════════════════════════════════════════════════════════════════════

TIER_TO_NUM = {"Baseline": 1, "Mild": 2, "Moderate": 3, "Critical": 4}

# Map tier columns to ordinal numbers
for col, num_col in [("t5_young_tier",  "YOUNG_NUM4"),
                     ("t5_cooper_tier", "COOPER_NUM4"),
                     ("t5_dissco_tier", "DISSCO_NUM4"),
                     ("t5_mosaic_tier", "MOSAIC_FROZEN_NUM4"),
                     ("LLM_FOUR_TIER",  "LLM_NUM4"),
                     ("OW_FOUR_TIER",   "OW_NUM4"),
                     # assessor columns
                     ("LLM_ASSESSOR_A_TIER", "CW_A_NUM4"),
                     ("LLM_ASSESSOR_B_TIER", "CW_B_NUM4"),
                     ("OW_ASSESSOR_A_TIER",  "OW_A_NUM4"),
                     ("OW_ASSESSOR_B_TIER",  "OW_B_NUM4")]:
    if col in results.columns:
        results[num_col] = results[col].map(TIER_TO_NUM)
        print(f"  {num_col}: {results[num_col].notna().sum():,} mapped, "
              f"{results[num_col].isna().sum():,} null")
    else:
        print(f"  ⚠️  {col} not found — {num_col} skipped")

df_cw = results[results["LLM_AVAILABLE"] == True].copy()
df_ow = results[results["OW_AVAILABLE"]  == True].copy()

# ── Helpers ────────────────────────────────────────────────────────────────────
def interpret_kappa(k):
    if k < 0:    return "Poor"
    elif k < .20: return "Slight"
    elif k < .40: return "Fair"
    elif k < .60: return "Moderate"
    elif k < .80: return "Substantial"
    else:         return "Almost perfect"

def bootstrap_kappa(y1, y2, weights="quadratic", B=2000):
    np.random.seed(42)
    kappas = []
    for _ in range(B):
        idx = np.random.choice(len(y1), len(y1), replace=True)
        try: kappas.append(cohen_kappa_score(y1[idx], y2[idx], weights=weights))
        except: pass
    return np.percentile(kappas, [2.5, 97.5])

def run_kappa_pairs(pairs):
    rows, cm_dict = [], {}
    for label, col1, col2, df_in, cohort_label in pairs:
        df_v = df_in[[col1, col2]].dropna()
        if len(df_v) < 5:
            print(f"  ⚠️  {label}: only {len(df_v)} rows — skipping"); continue
        y1 = df_v[col1].astype(int).values
        y2 = df_v[col2].astype(int).values
        k_qw             = cohen_kappa_score(y1, y2, weights="quadratic")
        k_lin            = cohen_kappa_score(y1, y2, weights="linear")
        k_raw            = cohen_kappa_score(y1, y2)
        ci_lo, ci_hi     = bootstrap_kappa(y1, y2)
        rows.append({"cohort": cohort_label, "comparison": label, "n": len(y1),
                     "kappa_quadratic":        round(k_qw,  4),
                     "kappa_linear":           round(k_lin, 4),
                     "kappa_unweighted":       round(k_raw, 4),
                     "ci_lo_bootstrap":        round(ci_lo, 4),
                     "ci_hi_bootstrap":        round(ci_hi, 4),
                     "exact_agreement_pct":    round(float(np.mean(y1==y2))*100, 2),
                     "adjacent_agreement_pct": round(float(np.mean(np.abs(y1-y2)<=1))*100, 2),
                     "interpretation":         interpret_kappa(k_qw)})
        cm = confusion_matrix(y1, y2, labels=[1,2,3,4])
        cm_dict[(label, cohort_label)] = cm
    return rows, cm_dict

# ── Classifier vs GT / LLM pairs ──────────────────────────────────────────────
kappa_pairs_ow = [
    ("Open-Weight vs Cooper GT",     "OW_NUM4",            "COOPER_NUM4",        df_ow, f"N={n_ow}"),
    ("Open-Weight vs Young GT",      "OW_NUM4",            "YOUNG_NUM4",         df_ow, f"N={n_ow}"),
    ("Open-Weight vs DiSSCo GT",     "OW_NUM4",            "DISSCO_NUM4",        df_ow, f"N={n_ow}"),
    ("Open-Weight vs MOSAIC Frozen", "OW_NUM4",            "MOSAIC_FROZEN_NUM4", df_ow, f"N={n_ow}"),
    ("Cooper GT vs Young GT",        "COOPER_NUM4",        "YOUNG_NUM4",         df_ow, f"N={n_ow}"),
    ("Cooper GT vs DiSSCo GT",       "COOPER_NUM4",        "DISSCO_NUM4",        df_ow, f"N={n_ow}"),
    ("Young GT vs DiSSCo GT",        "YOUNG_NUM4",         "DISSCO_NUM4",        df_ow, f"N={n_ow}"),
    ("MOSAIC Frozen vs Cooper GT",   "MOSAIC_FROZEN_NUM4", "COOPER_NUM4",        df_ow, f"N={n_ow}"),
    ("MOSAIC Frozen vs Young GT",    "MOSAIC_FROZEN_NUM4", "YOUNG_NUM4",         df_ow, f"N={n_ow}"),
    ("MOSAIC Frozen vs DiSSCo GT",   "MOSAIC_FROZEN_NUM4", "DISSCO_NUM4",        df_ow, f"N={n_ow}"),
]
kappa_pairs_cw = [
    ("Closed-Weight vs Cooper GT",     "LLM_NUM4",           "COOPER_NUM4",        df_cw, f"N={n_cw}"),
    ("Closed-Weight vs Young GT",      "LLM_NUM4",           "YOUNG_NUM4",         df_cw, f"N={n_cw}"),
    ("Closed-Weight vs DiSSCo GT",     "LLM_NUM4",           "DISSCO_NUM4",        df_cw, f"N={n_cw}"),
    ("Closed-Weight vs Open-Weight",   "LLM_NUM4",           "OW_NUM4",            df_cw, f"N={n_cw}"),
    ("Closed-Weight vs MOSAIC Frozen", "LLM_NUM4",           "MOSAIC_FROZEN_NUM4", df_cw, f"N={n_cw}"),
    ("Cooper GT vs Young GT",          "COOPER_NUM4",        "YOUNG_NUM4",         df_cw, f"N={n_cw}"),
    ("Cooper GT vs DiSSCo GT",         "COOPER_NUM4",        "DISSCO_NUM4",        df_cw, f"N={n_cw}"),
    ("Young GT vs DiSSCo GT",          "YOUNG_NUM4",         "DISSCO_NUM4",        df_cw, f"N={n_cw}"),
    ("MOSAIC Frozen vs Cooper GT",     "MOSAIC_FROZEN_NUM4", "COOPER_NUM4",        df_cw, f"N={n_cw}"),
    ("MOSAIC Frozen vs Young GT",      "MOSAIC_FROZEN_NUM4", "YOUNG_NUM4",         df_cw, f"N={n_cw}"),
    ("MOSAIC Frozen vs DiSSCo GT",     "MOSAIC_FROZEN_NUM4", "DISSCO_NUM4",        df_cw, f"N={n_cw}"),
    ("Open-Weight vs MOSAIC Frozen",   "OW_NUM4",            "MOSAIC_FROZEN_NUM4", df_cw, f"N={n_cw}"),
]

kappa_rows, cm_data = run_kappa_pairs(kappa_pairs_ow + kappa_pairs_cw)

# ── Inter-assessor pairs ───────────────────────────────────────────────────────
assessor_pairs = [
    ("CW: GPT-4o vs DeepSeek",   "CW_A_NUM4", "CW_B_NUM4", df_cw, f"N={n_cw}"),
    ("OW: Gemma vs Qwen",        "OW_A_NUM4", "OW_B_NUM4", df_ow, f"N={n_ow}"),
    ("CW: GPT-4o vs Final",      "CW_A_NUM4", "LLM_NUM4",  df_cw, f"N={n_cw}"),
    ("CW: DeepSeek vs Final",    "CW_B_NUM4", "LLM_NUM4",  df_cw, f"N={n_cw}"),
    ("OW: Gemma vs Final",       "OW_A_NUM4", "OW_NUM4",   df_ow, f"N={n_ow}"),
    ("OW: Qwen vs Final",        "OW_B_NUM4", "OW_NUM4",   df_ow, f"N={n_ow}"),
]
assessor_kappa_rows, assessor_cm_data = run_kappa_pairs(assessor_pairs)

# ── Print summary ──────────────────────────────────────────────────────────────
def print_kappa_summary(rows, title):
    print(f"\n{'='*78}")
    print(f"  {title}")
    print(f"{'='*78}")
    print(f"{'Comparison':<35} {'Cohort':>8} {'κw':>7} {'95% CI':>18} "
          f"{'Exact%':>7} {'Adj%':>6}  Interp.")
    print(f"{'─'*78}")
    for r in rows:
        ci = f"[{r['ci_lo_bootstrap']:.3f}–{r['ci_hi_bootstrap']:.3f}]"
        print(f"{r['comparison']:<35} {r['cohort']:>8} "
              f"{r['kappa_quadratic']:>7.4f} {ci:>18} "
              f"{r['exact_agreement_pct']:>6.1f}% {r['adjacent_agreement_pct']:>5.1f}%  "
              f"{r['interpretation']}")
    print(f"{'='*78}")

print_kappa_summary(kappa_rows, "Classifier vs GT / LLM Agreement")
print_kappa_summary(assessor_kappa_rows, "Inter-Assessor Agreement")

# ── Delta consolidation ────────────────────────────────────────────────────────
print("\n── Consolidation value (δ = κw_consolidated − avg κw_assessors_vs_final) ──")
for pipeline, a_lbl, b_lbl, consol_lbl in [
    ("CW", "CW: GPT-4o vs Final", "CW: DeepSeek vs Final", "Closed-Weight vs Cooper GT"),
    ("OW", "OW: Gemma vs Final",  "OW: Qwen vs Final",     "Open-Weight vs Cooper GT"),
]:
    kw_consol = next((r["kappa_quadratic"] for r in kappa_rows
                      if r["comparison"] == consol_lbl), None)
    kw_a = next((r["kappa_quadratic"] for r in assessor_kappa_rows
                 if r["comparison"] == a_lbl), None)
    kw_b = next((r["kappa_quadratic"] for r in assessor_kappa_rows
                 if r["comparison"] == b_lbl), None)
    if all(v is not None for v in [kw_consol, kw_a, kw_b]):
        avg  = (kw_a + kw_b) / 2
        d    = kw_consol - avg
        v    = ("Adds value" if d > 0.02 else "Marginal" if d > 0 else "Assessors already agree well")
        print(f"  {pipeline}: avg assessor κw={avg:.3f}, consolidated κw={kw_consol:.3f}, "
              f"δ={d:+.3f}  → {v}")

# ── Save ───────────────────────────────────────────────────────────────────────
all_kappa = pd.DataFrame(kappa_rows + assessor_kappa_rows)
all_kappa.to_csv(f"{RESULTS_DIR}/kappa_table.csv", index=False)
display(all_kappa)
print(f"\n💾 kappa_table: {len(all_kappa)} rows → {RESULTS_DIR}/kappa_table.csv")

# Confusion matrix long-form CSV
cm_all_rows = []
for (label, cohort), cm_arr in {**cm_data, **assessor_cm_data}.items():
    cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(1)
    for i in range(4):
        for j in range(4):
            cm_all_rows.append({"cohort": cohort, "comparison": label,
                                 "true_tier":  TIER_ORDER[i],
                                 "pred_tier":  TIER_ORDER[j],
                                 "count":      int(cm_arr[i,j]),
                                 "proportion": round(float(cm_norm[i,j]), 4)})
cm_long = pd.DataFrame(cm_all_rows)
cm_long.to_csv(f"{RESULTS_DIR}/confusion_matrices.csv", index=False)
print(f"💾 confusion_matrices: {len(cm_long)} rows → {RESULTS_DIR}/confusion_matrices.csv")
print("\n✅ Kappa analysis complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12 — Survival Plots (KM, NA, CIF) + Love Plots
# Inputs:  df_cw, df_ow, results, TIER_ORDER, CLASSIFIERS, ENC_GROUPS
# Outputs: display + /content/results/*.pdf/png + km_survival_checkpoints.csv
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

# ── Aesthetics (updated for short tier names) ──────────────────────────────────
TIER_COLORS = {
    "Baseline": "#2196F3",
    "Mild":     "#4CAF50",
    "Moderate": "#FF9800",
    "Critical": "#F44336",
}
TIER_LS = {
    "Baseline": "solid",
    "Mild":     "dashed",
    "Moderate": "dashdot",
    "Critical": "dotted",
}
CLASSIFIER_DISPLAY = {
    "Young_GT_CW":  "Young GT",    "Young_GT_OW":  "Young GT",
    "Cooper_GT_CW": "Cooper GT",   "Cooper_GT_OW": "Cooper GT",
    "CW_LLM":       "Closed-Weight",
    "OW_LLM_CW":    "Open-Weight", "OW_LLM":       "Open-Weight",
    "DiSSCo_OW":    "DiSSCo GT",
}
def clean_label(lbl):
    return CLASSIFIER_DISPLAY.get(lbl, lbl.replace("_", " "))

TAU = 1825   # 5-year horizon in days

def _save_fig(fig, stem):
    for ext in ["pdf","png"]:
        fig.savefig(f"{RESULTS_DIR}/{stem}.{ext}", bbox_inches="tight", dpi=300)
    print(f"  💾 {stem}.pdf / .png")

# ── Outcome definitions ────────────────────────────────────────────────────────
KM_OUTCOMES  = [("All-cause mortality", "OS_TIME", "OS_EVENT")]
CIF_OUTCOMES = [
    ("New complication",    "COMP_TIME_DAYS",        "COMP_EVENT"),
    ("Emergency encounter", "DAYS_EMERGENCY",         "EVENT_EMERGENCY"),
    ("Inpatient admission", "DAYS_INPATIENT_SURGICAL","EVENT_INPATIENT_SURGICAL"),
]

# ── Figure groups ──────────────────────────────────────────────────────────────
FIG_GROUPS = {
    "fig_A_young_cooper_ow":     (df_ow, [("Young GT",      "t5_young_tier"),
                                           ("Cooper GT",     "t5_cooper_tier"),
                                           ("Open-Weight",   "OW_FOUR_TIER")]),
    "fig_B_young_cooper_dissco": (df_ow, [("Young GT",      "t5_young_tier"),
                                           ("Cooper GT",     "t5_cooper_tier"),
                                           ("DiSSCo GT",     "t5_dissco_tier")]),
    "fig_C_mosaic_ow":           (df_ow, [("MOSAIC Frozen",  "t5_mosaic_tier"),
                                           ("Open-Weight",   "OW_FOUR_TIER")]),
    "fig_D_sensitivity_cw":      (df_cw, [("Young GT",      "t5_young_tier"),
                                           ("Cooper GT",     "t5_cooper_tier"),
                                           ("Open-Weight",   "OW_FOUR_TIER"),
                                           ("Closed-Weight", "LLM_FOUR_TIER")]),
    "fig_E_all_gts":             (df_ow, [("Young GT",      "t5_young_tier"),
                                           ("Cooper GT",     "t5_cooper_tier"),
                                           ("DiSSCo GT",     "t5_dissco_tier"),
                                           ("MOSAIC Frozen", "t5_mosaic_tier")]),
    "fig_F_ow_cooper":           (df_ow, [("Open-Weight",   "OW_FOUR_TIER"),
                                           ("Cooper GT",     "t5_cooper_tier")]),
}

# ── Helpers ────────────────────────────────────────────────────────────────────
def make_aj_event(df, event_col, competing_col="OS_EVENT"):
    aj = pd.Series(0, index=df.index)
    aj[df[event_col] == 1] = 1
    aj[(df[event_col] == 0) & (df[competing_col] == 1)] = 2
    return aj

def logrank_p(df, tier_col, time_col, event_col):
    sub = df.dropna(subset=[tier_col])
    try:
        res = multivariate_logrank_test(sub[time_col], sub[tier_col], sub[event_col])
        p   = res.p_value
        return "p < 0.001" if p < 0.001 else (f"p = {p:.3f}" if p < 0.01 else f"p = {p:.2f}")
    except Exception:
        return ""

# ── Panel builders ─────────────────────────────────────────────────────────────
def plot_km_panel(ax, df, tier_col, time_col, event_col, title=""):
    sub = df.dropna(subset=[tier_col]).copy()
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        kmf = KaplanMeierFitter(label=tier)
        kmf.fit(sub.loc[mask, time_col], sub.loc[mask, event_col],
                timeline=np.linspace(0, TAU, 500))
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            color=TIER_COLORS.get(tier, "grey"),
            linestyle=TIER_LS.get(tier, "solid"),
            linewidth=1.8, ci_alpha=0.12,
            show_censors=True, censor_styles={"ms":3,"marker":"|"},
            label=tier)
    ax.set_title(f"{title}\n{logrank_p(sub, tier_col, time_col, event_col)}",
                 fontsize=9, pad=4)
    ax.set_xlabel("Days from T₅", fontsize=8)
    ax.set_ylabel("Survival probability", fontsize=8)
    ax.set_xlim(0, TAU); ax.set_ylim(0, 1.05)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6.5, framealpha=0.5, loc="lower left")

def plot_na_panel(ax, df, tier_col, time_col, event_col, title=""):
    sub = df.dropna(subset=[tier_col]).copy()
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        naf = NelsonAalenFitter(label=tier)
        naf.fit(sub.loc[mask, time_col], sub.loc[mask, event_col],
                timeline=np.linspace(0, TAU, 500))
        naf.plot_cumulative_hazard(
            ax=ax, ci_show=True,
            color=TIER_COLORS.get(tier, "grey"),
            linestyle=TIER_LS.get(tier, "solid"),
            linewidth=1.8, ci_alpha=0.12, label=tier)
    ax.set_title(f"{title}\n{logrank_p(sub, tier_col, time_col, event_col)}",
                 fontsize=9, pad=4)
    ax.set_xlabel("Days from T₅", fontsize=8)
    ax.set_ylabel("Cumulative hazard H(t)", fontsize=8)
    ax.set_xlim(0, TAU)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6.5, framealpha=0.5, loc="upper left")

def plot_cif_panel(ax, df, tier_col, time_col, event_col, title=""):
    sub = df.dropna(subset=[tier_col]).copy()
    sub["_AJ"] = make_aj_event(sub, event_col, "OS_EVENT")
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        try:
            ajf = AalenJohansenFitter(calculate_variance=True)
            ajf.fit(sub.loc[mask, time_col], sub.loc[mask,"_AJ"], event_of_interest=1)
            t_v = ajf.cumulative_density_.index.values
            c_v = ajf.cumulative_density_["CIF_1"].values
            ax.step(t_v, c_v, where="post",
                    color=TIER_COLORS.get(tier,"grey"),
                    linestyle=TIER_LS.get(tier,"solid"),
                    linewidth=1.8, label=tier)
            if ajf.variance_ is not None:
                se = pd.Series(np.sqrt(np.clip(ajf.variance_.values, 0, None)),
                               index=ajf.variance_.index)
                se_a = se.reindex(t_v, method="ffill").fillna(0).values
                ax.fill_between(t_v,
                                np.clip(c_v - 1.96*se_a, 0, 1),
                                np.clip(c_v + 1.96*se_a, 0, 1),
                                step="post", alpha=0.12,
                                color=TIER_COLORS.get(tier,"grey"))
        except Exception as e:
            print(f"    AJ failed {tier}: {e}")
    ax.set_title(title, fontsize=9, pad=4)
    ax.set_xlabel("Days from T₅", fontsize=8)
    ax.set_ylabel("Cumulative incidence F(t)", fontsize=8)
    ax.set_xlim(0, TAU); ax.set_ylim(0, 1.0)
    ax.set_xticks([0, 365, 730, 1095, 1460, 1825])
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6.5, framealpha=0.5, loc="upper left")

# ── Master figure builder ──────────────────────────────────────────────────────
def build_survival_figure(fig_stem, df, classifiers):
    n_cls = len(classifiers)

    # KM + NA
    fig_km, axes_km = plt.subplots(n_cls, 2, figsize=(10, 4*n_cls), squeeze=False)
    fig_km.suptitle(f"KM & Nelson–Aalen — All-cause Mortality\n{fig_stem}",
                    fontsize=11, y=1.01)
    for row, (label, tier_col) in enumerate(classifiers):
        if tier_col not in df.columns:
            print(f"  ⚠ {label} — {tier_col} missing"); continue
        plot_km_panel(axes_km[row,0], df, tier_col, "OS_TIME","OS_EVENT",
                      title=f"{label} — KM")
        plot_na_panel(axes_km[row,1], df, tier_col, "OS_TIME","OS_EVENT",
                      title=f"{label} — Nelson-Aalen")
    fig_km.tight_layout()
    _save_fig(fig_km, f"{fig_stem}_km_na")
    plt.show(); plt.close(fig_km)

    # CIF
    fig_cif, axes_cif = plt.subplots(n_cls, 3, figsize=(14, 4*n_cls), squeeze=False)
    fig_cif.suptitle(f"Cumulative Incidence (Aalen–Johansen)\n{fig_stem}",
                     fontsize=11, y=1.01)
    for row, (label, tier_col) in enumerate(classifiers):
        if tier_col not in df.columns: continue
        for col, (out_lbl, tcol, ecol) in enumerate(CIF_OUTCOMES):
            if tcol not in df.columns or ecol not in df.columns:
                axes_cif[row,col].set_visible(False); continue
            plot_cif_panel(axes_cif[row,col], df, tier_col, tcol, ecol,
                           title=f"{label}\n{out_lbl}")
    fig_cif.tight_layout()
    _save_fig(fig_cif, f"{fig_stem}_cif")
    plt.show(); plt.close(fig_cif)
    print(f"  ✅ {fig_stem} complete")

# ── Build all survival figures ─────────────────────────────────────────────────
print("Building survival figures …\n")
for fig_stem, (df_in, classifiers) in FIG_GROUPS.items():
    print(f"  → {fig_stem}")
    build_survival_figure(fig_stem, df_in, classifiers)

# ── Side-by-side OW vs Cooper KM (user request) ───────────────────────────────
print("\nBuilding side-by-side OW vs Cooper KM …")

fig_sb, axes_sb = plt.subplots(1, 2, figsize=(12, 5),
                                gridspec_kw={"wspace": 0.35})
fig_sb.suptitle(
    f"All-cause Mortality: Open-Weight vs Cooper GT  —  OW Cohort (N={n_ow:,})\n"
    "Kaplan–Meier with 95% CI",
    fontsize=12, fontweight="bold", y=1.02)

plot_km_panel(axes_sb[0], df_ow, "OW_FOUR_TIER",   "OS_TIME","OS_EVENT",
              title="Open-Weight LLM")
plot_km_panel(axes_sb[1], df_ow, "t5_cooper_tier", "OS_TIME","OS_EVENT",
              title="Cooper GT")

# Align y-axes for fair comparison
y_lo = min(ax.get_ylim()[0] for ax in axes_sb)
for ax in axes_sb:
    ax.set_ylim(y_lo, 1.05)

fig_sb.tight_layout()
_save_fig(fig_sb, "km_ow_vs_cooper_sidebyside")
plt.show(); plt.close(fig_sb)
print("  ✅ Side-by-side OW vs Cooper saved")

# ── SMD cache for love plots ───────────────────────────────────────────────────
COVARIATE_ORDER = [
    "New complication",    "Death in follow-up",  "Microvascular disease",
    "Macrovascular disease","Insulin use at T5",  "Neuropathy at T5",
    "Nephropathy at T5",   "Retinopathy at T5",   "Insulin (first drug)",
    "Female sex",          "Follow-up time",       "DCSI score at T5",
    "Age at T5",
]

def build_smd_dict(df_in, tier_col, tier_order, ref_tier="Baseline"):
    df  = df_in.dropna(subset=[tier_col]).copy()
    df  = df[df[tier_col].isin(tier_order)]
    smd = {}
    _map = {
        "Age at T5":           ("AGE_AT_T5",    "continuous"),
        "DCSI score at T5":    ("t5_dcsi",      "continuous"),
        "Follow-up time":      ("OS_TIME",      "continuous"),
        "Female sex":          (None,           "binary_gender"),
        "Insulin (first drug)":(None,           "binary_drug"),
        "Retinopathy at T5":   ("t5_ret",       "binary"),
        "Nephropathy at T5":   ("t5_neph",      "binary"),
        "Neuropathy at T5":    ("t5_neu",       "binary"),
        "Insulin use at T5":   ("t5_insulin",   "binary"),
        "Macrovascular disease":(None,          "macro"),
        "Microvascular disease":(None,          "micro"),
        "Death in follow-up":  ("DIED_IN_FOLLOWUP","binary"),
        "New complication":    ("COMP_EVENT",   "binary"),
    }
    for cov, (col, kind) in _map.items():
        try:
            if kind == "continuous" and col in df.columns:
                smd[cov] = compute_smd_continuous(df[col], df[tier_col], ref_tier)
            elif kind == "binary" and col in df.columns:
                smd[cov] = compute_smd_binary(df[col].fillna(0).astype(int),
                                              df[tier_col], ref_tier)
            elif kind == "binary_gender" and "GENDER" in df.columns:
                smd[cov] = compute_smd_binary((df["GENDER"]=="F").astype(int),
                                              df[tier_col], ref_tier)
            elif kind == "binary_drug" and "FIRST_DRUG_CLASS" in df.columns:
                smd[cov] = compute_smd_binary((df["FIRST_DRUG_CLASS"]=="Insulin").astype(int),
                                              df[tier_col], ref_tier)
            elif kind == "macro":
                macro = df[["t5_cvd","t5_cbv","t5_pvd"]].fillna(0).max(axis=1)
                smd[cov] = compute_smd_binary(macro, df[tier_col], ref_tier)
            elif kind == "micro":
                micro = df[["t5_ret","t5_neph","t5_neu"]].fillna(0).max(axis=1)
                smd[cov] = compute_smd_binary(micro, df[tier_col], ref_tier)
        except Exception:
            pass
    return smd

print("\nBuilding SMD cache …")
smd_cache = {}
for df_in, tier_col, label in CLASSIFIERS:
    if tier_col not in df_in.columns: continue
    smd_cache[label] = build_smd_dict(df_in, tier_col, TIER_ORDER)
    print(f"  ✅ {label}: {len(smd_cache[label])} covariates")

# ── Love plots ─────────────────────────────────────────────────────────────────
def build_love_plots():
    SMD_CAP = 2.0
    non_ref = [t for t in TIER_ORDER if t != "Baseline"]
    cohort_groups = {
        f"OW Cohort (N={n_ow:,})": [(label, smd_cache[label])
                                     for df_in, _, label in CLASSIFIERS
                                     if df_in is df_ow and label in smd_cache],
        f"CW Cohort (N={n_cw:,})": [(label, smd_cache[label])
                                     for df_in, _, label in CLASSIFIERS
                                     if df_in is df_cw and label in smd_cache],
    }
    for cohort_name, entries in cohort_groups.items():
        if not entries: continue
        n_cls = len(entries)
        fig, axes = plt.subplots(1, n_cls, figsize=(5*n_cls, 7), squeeze=False)
        fig.suptitle(
            f"Love Plots — Standardised Mean Differences\n"
            f"{cohort_name}  (ref = Baseline, dashed = |SMD| > 0.10)",
            fontsize=11)
        for col, (label, smd_dict) in enumerate(entries):
            ax   = axes[0, col]
            covs = [c for c in COVARIATE_ORDER if c in smd_dict]
            for i, cov in enumerate(covs):
                for tier in non_ref:
                    val = smd_dict[cov].get(tier, np.nan)
                    if np.isnan(val): continue
                    capped   = val > SMD_CAP
                    val_plot = min(val, SMD_CAP)
                    ax.scatter(val_plot, i,
                               color=TIER_COLORS.get(tier, "grey"),
                               zorder=3, s=40,
                               marker=">" if capped else "o",
                               label=tier if i == 0 else "_nolegend_")
            ax.axvline(0.10, color="red",    linestyle="--", lw=1,   alpha=0.7)
            ax.axvline(0.25, color="orange", linestyle=":",  lw=0.8, alpha=0.7)
            ax.axvline(0.00, color="grey",   linestyle="-",  lw=0.6, alpha=0.4)
            ax.set_yticks(range(len(covs)))
            ax.set_yticklabels(covs, fontsize=7.5)
            ax.set_xlabel("Standardised Mean Difference", fontsize=8)
            ax.set_xlim(-0.05, SMD_CAP + 0.1)
            ax.set_title(clean_label(label), fontsize=9)
            ax.tick_params(labelsize=7)
            n_capped = sum(1 for c in covs for t in non_ref
                           if smd_dict.get(c,{}).get(t,0) > SMD_CAP)
            if n_capped:
                ax.text(0.98, 0.02, f"▶ = SMD > {SMD_CAP} ({n_capped} values)",
                        transform=ax.transAxes, fontsize=6,
                        ha="right", va="bottom", color="grey")
            if col == 0:
                ax.legend(fontsize=6.5, loc="lower right")
        fig.tight_layout()
        stem = "love_plot_ow" if "OW" in cohort_name else "love_plot_cw"
        _save_fig(fig, stem)
        plt.show(); plt.close(fig)
        print(f"  ✅ Love plot — {cohort_name}")

print("\nBuilding love plots …")
build_love_plots()

# ── Zoomed side-by-side OW vs Cooper KM ───────────────────────────────────────
def plot_km_panel_zoomed(ax, df, tier_col, time_col, event_col, title="", y_lo=0.75):
    """Same as plot_km_panel but with zoomed y-axis and cleaner styling."""
    sub = df.dropna(subset=[tier_col]).copy()
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        kmf = KaplanMeierFitter(label=tier)
        kmf.fit(sub.loc[mask, time_col], sub.loc[mask, event_col],
                timeline=np.linspace(0, TAU, 500))
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            color=TIER_COLORS.get(tier, "grey"),
            linestyle=TIER_LS.get(tier, "solid"),
            linewidth=2.0, ci_alpha=0.15,
            show_censors=True, censor_styles={"ms": 4, "marker": "|"},
            label=tier)

    p_str = logrank_p(sub, tier_col, time_col, event_col)
    ax.set_title(f"{title}\n{p_str}", fontsize=10, pad=6)
    ax.set_xlabel("Days from T₅", fontsize=9)
    ax.set_ylabel("Survival probability", fontsize=9)
    ax.set_xlim(0, TAU)
    ax.set_ylim(y_lo, 1.02)
    ax.set_xticks([0, 365, 730, 1095, 1460, 1825])
    ax.set_xticklabels(["0","1yr","2yr","3yr","4yr","5yr"], fontsize=8)
    ax.yaxis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(lambda y, _: f"{y:.2f}"))
    ax.tick_params(labelsize=8)
    ax.grid(axis="y", linestyle=":", alpha=0.4, color="grey")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, framealpha=0.6, loc="lower left",
              handlelength=2.5, borderpad=0.8)

# Determine shared y floor from actual data
def _km_floor(df, tier_col, time_col, event_col, padding=0.03):
    sub = df.dropna(subset=[tier_col])
    mins = []
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        kmf = KaplanMeierFitter()
        kmf.fit(sub.loc[mask, time_col], sub.loc[mask, event_col])
        mins.append(float(kmf.survival_function_.iloc[:, 0].min()))
    return max(0.0, min(mins) - padding) if mins else 0.75

# Compute shared floor across both classifiers
y_floor = min(
    _km_floor(df_ow, "OW_FOUR_TIER",   "OS_TIME", "OS_EVENT"),
    _km_floor(df_ow, "t5_cooper_tier", "OS_TIME", "OS_EVENT"),
)
y_floor = round(y_floor * 20) / 20   # snap to nearest 0.05

print(f"Shared y-floor: {y_floor:.2f}")

fig_z, axes_z = plt.subplots(1, 2, figsize=(13, 5.5),
                              gridspec_kw={"wspace": 0.32})
fig_z.suptitle(
    f"All-cause Mortality: Open-Weight vs Cooper GT  —  OW Cohort (N={n_ow:,})\n"
    "Kaplan–Meier with 95% CI  (y-axis zoomed to event range)",
    fontsize=12, fontweight="bold", y=1.02)

plot_km_panel_zoomed(axes_z[0], df_ow, "OW_FOUR_TIER",   "OS_TIME", "OS_EVENT",
                     title="Open-Weight LLM", y_lo=y_floor)
plot_km_panel_zoomed(axes_z[1], df_ow, "t5_cooper_tier", "OS_TIME", "OS_EVENT",
                     title="Cooper GT",       y_lo=y_floor)

# Remove duplicate y-label on right panel
axes_z[1].set_ylabel("")

fig_z.tight_layout()
_save_fig(fig_z, "km_ow_vs_cooper_zoomed")
plt.show()
plt.close(fig_z)
print("✅ Zoomed KM plot saved")

# ── Zoomed CIF: OW vs Cooper, New Complication + Emergency only ────────────────
def _cif_y_ceil(df, tier_col, time_col, event_col, padding=0.05):
    """Get the max CIF value across all tiers for y-axis ceiling."""
    sub = df.dropna(subset=[tier_col]).copy()
    sub["_AJ"] = make_aj_event(sub, event_col, "OS_EVENT")
    maxes = []
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        try:
            ajf = AalenJohansenFitter(calculate_variance=True)
            ajf.fit(sub.loc[mask, time_col], sub.loc[mask,"_AJ"], event_of_interest=1)
            maxes.append(float(ajf.cumulative_density_["CIF_1"].max()))
        except Exception:
            pass
    return min(1.0, max(maxes) + padding) if maxes else 1.0

def plot_cif_panel_zoomed(ax, df, tier_col, time_col, event_col, title="", y_ceil=None):
    sub = df.dropna(subset=[tier_col]).copy()
    sub["_AJ"] = make_aj_event(sub, event_col, "OS_EVENT")
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        try:
            ajf = AalenJohansenFitter(calculate_variance=True)
            ajf.fit(sub.loc[mask, time_col], sub.loc[mask,"_AJ"], event_of_interest=1)
            t_v = ajf.cumulative_density_.index.values
            c_v = ajf.cumulative_density_["CIF_1"].values
            ax.step(t_v, c_v, where="post",
                    color=TIER_COLORS.get(tier,"grey"),
                    linestyle=TIER_LS.get(tier,"solid"),
                    linewidth=2.0, label=tier)
            if ajf.variance_ is not None:
                se   = pd.Series(np.sqrt(np.clip(ajf.variance_.values,0,None)),
                                 index=ajf.variance_.index)
                se_a = se.reindex(t_v, method="ffill").fillna(0).values
                ax.fill_between(t_v,
                                np.clip(c_v - 1.96*se_a, 0, 1),
                                np.clip(c_v + 1.96*se_a, 0, 1),
                                step="post", alpha=0.12,
                                color=TIER_COLORS.get(tier,"grey"))
        except Exception as e:
            print(f"    AJ failed {tier}: {e}")

    # Log-rank p
    try:
        sub_ev = sub.copy()
        lr    = multivariate_logrank_test(sub_ev[time_col], sub_ev[tier_col],
                                          (sub_ev["_AJ"]==1).astype(int))
        p_val = lr.p_value
        p_str = "p < 0.001" if p_val < 0.001 else (f"p = {p_val:.3f}" if p_val < 0.01
                                                     else f"p = {p_val:.2f}")
    except Exception:
        p_str = ""

    ax.set_title(f"{title}\n{p_str}", fontsize=10, pad=6)
    ax.set_xlabel("Days from T₅", fontsize=9)
    ax.set_ylabel("Cumulative incidence F(t)", fontsize=9)
    ax.set_xlim(0, TAU)
    ax.set_ylim(0, y_ceil if y_ceil else 1.0)
    ax.set_xticks([0, 365, 730, 1095, 1460, 1825])
    ax.set_xticklabels(["0","1yr","2yr","3yr","4yr","5yr"], fontsize=8)
    ax.tick_params(labelsize=8)
    ax.grid(axis="y", linestyle=":", alpha=0.4, color="grey")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, framealpha=0.6, loc="upper left",
              handlelength=2.5, borderpad=0.8)

# Outcomes to show (no inpatient)
CIF_ZOOMED = [
    ("New complication",    "COMP_TIME_DAYS", "COMP_EVENT"),
    ("Emergency encounter", "DAYS_EMERGENCY", "EVENT_EMERGENCY"),
]

CLASSIFIERS_ZOOMED = [
    ("Open-Weight", "OW_FOUR_TIER",   df_ow),
    ("Cooper GT",   "t5_cooper_tier", df_ow),
]

# Compute shared y-ceiling per outcome column (same scale across OW and Cooper rows)
y_ceils = {}
for out_label, time_col, event_col in CIF_ZOOMED:
    ceils = [_cif_y_ceil(df, tc, time_col, event_col)
             for _, tc, df in CLASSIFIERS_ZOOMED
             if tc in df.columns]
    raw   = max(ceils) if ceils else 1.0
    y_ceils[out_label] = round(raw * 20) / 20 + 0.05   # snap to 0.05

print("Y-axis ceilings per outcome:")
for k, v in y_ceils.items():
    print(f"  {k}: {v:.2f}")

# Build figure: 2 rows × 2 cols
fig_cif_z, axes_cif_z = plt.subplots(
    2, 2, figsize=(12, 9),
    gridspec_kw={"hspace": 0.45, "wspace": 0.32})

fig_cif_z.suptitle(
    f"Cumulative Incidence (Aalen–Johansen): Open-Weight vs Cooper GT\n"
    f"OW Cohort (N={n_ow:,})  —  y-axis zoomed to event range",
    fontsize=12, fontweight="bold", y=1.02)

for row, (cls_label, tier_col, df_in) in enumerate(CLASSIFIERS_ZOOMED):
    for col, (out_label, time_col, event_col) in enumerate(CIF_ZOOMED):
        ax = axes_cif_z[row, col]
        if tier_col not in df_in.columns or time_col not in df_in.columns:
            ax.set_visible(False); continue

        # Only show y-label on left column, x-label on bottom row
        plot_cif_panel_zoomed(
            ax, df_in, tier_col, time_col, event_col,
            title=f"{cls_label}\n{out_label}",
            y_ceil=y_ceils[out_label])

        if col > 0:
            ax.set_ylabel("")
        if row < len(CLASSIFIERS_ZOOMED) - 1:
            ax.set_xlabel("")

fig_cif_z.tight_layout()
_save_fig(fig_cif_z, "cif_ow_vs_cooper_zoomed")
plt.show()
plt.close(fig_cif_z)
print("✅ Zoomed CIF plot saved")

# ── KM survival checkpoints CSV ───────────────────────────────────────────────
print("\nSaving KM survival checkpoints …")
checkpoints = [0, 365, 730, 1095, 1460, 1825]
km_cp_rows  = []
for df_in, tier_col, label in [(df_in, tc, lbl) for df_in, tc, lbl in
                                [(row[0], row[1], row[2]) for row in
                                 [(df_in, tc, lbl) for df_in, tc, lbl in
                                  [(d, t, l) for d, t, l in
                                   [(df_in, tier_col, label)
                                    for df_in, tier_col, label in CLASSIFIERS]]]]]:
    pass  # cleaner loop below

km_cp_rows = []
for df_in, tier_col, label in CLASSIFIERS:
    if tier_col not in df_in.columns: continue
    sub = df_in.dropna(subset=[tier_col])
    for tier in TIER_ORDER:
        mask = sub[tier_col] == tier
        if mask.sum() < 5: continue
        kmf = KaplanMeierFitter()
        kmf.fit(sub.loc[mask,"OS_TIME"], sub.loc[mask,"OS_EVENT"])
        for t in checkpoints:
            try:
                sv = float(kmf.survival_function_at_times([t]).values[0])
                ci = kmf.confidence_interval_survival_function_at_times([t])
                lo, hi = float(ci.iloc[0,0]), float(ci.iloc[0,1])
            except Exception:
                sv, lo, hi = np.nan, np.nan, np.nan
            km_cp_rows.append({"classifier":label,"tier":tier,"days":t,
                                "km_survival":round(sv,4),
                                "ci_lower":round(lo,4),"ci_upper":round(hi,4)})

km_cp_df = pd.DataFrame(km_cp_rows)
km_cp_df.to_csv(f"{RESULTS_DIR}/km_survival_checkpoints.csv", index=False)
print(f"  💾 km_survival_checkpoints: {len(km_cp_df):,} rows")
print("\n✅ Cell 12 complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 13 — Results Summary Tables (KM, NA, CIF, SMD)
# Inputs:  FIG_GROUPS, smd_cache, COVARIATE_ORDER  (all from Cell 12)
# Outputs: display + /content/results/results_*_summary.csv
# ═══════════════════════════════════════════════════════════════════════════════
from IPython.display import display

TAU_CHECKPOINTS = [365, 730, 1095, 1460, 1825]
TAU_LABELS      = ["1yr","2yr","3yr","4yr","5yr"]

def _nearest(idx, t):
    i = idx.searchsorted(t, side="left")
    return min(i, len(idx)-1)

print("="*70)
print("RESULTS SUMMARY TABLES")
print("="*70)

# ── 1. KM survival ─────────────────────────────────────────────────────────────
print("\n── 1. Kaplan–Meier survival at checkpoints ──")
km_rows = []
for fig_stem, (df_in, classifiers) in FIG_GROUPS.items():
    for cls_label, tier_col in classifiers:
        if tier_col not in df_in.columns: continue
        sub = df_in.dropna(subset=[tier_col])
        try:
            lr    = multivariate_logrank_test(sub["OS_TIME"], sub[tier_col], sub["OS_EVENT"])
            p_val = lr.p_value
            p_str = "< 0.001" if p_val < 0.001 else f"{p_val:.3f}"
        except Exception:
            p_str = "—"
        print(f"\n  {fig_stem} | {cls_label}  (log-rank p {p_str})")
        print(f"  {'Tier':<22} {'N':>5} {'Events':>7}  " +
              "  ".join(f"{l:>18}" for l in TAU_LABELS))
        print("  " + "─"*100)
        for tier in TIER_ORDER:
            mask = sub[tier_col] == tier
            n    = mask.sum()
            if n < 5: continue
            kmf  = KaplanMeierFitter()
            kmf.fit(sub.loc[mask,"OS_TIME"], sub.loc[mask,"OS_EVENT"])
            ev   = int(sub.loc[mask,"OS_EVENT"].sum())
            vals = []
            for t in TAU_CHECKPOINTS:
                i  = _nearest(kmf.survival_function_.index, t)
                sv = kmf.survival_function_.iloc[i,0]
                lo = kmf.confidence_interval_.iloc[i,0]
                hi = kmf.confidence_interval_.iloc[i,1]
                vals.append(f"{sv:.3f} [{lo:.3f}–{hi:.3f}]")
                km_rows.append({"figure":fig_stem,"classifier":cls_label,"tier":tier,
                                 "n":n,"events":ev,"timepoint":t,
                                 "timepoint_label":TAU_LABELS[TAU_CHECKPOINTS.index(t)],
                                 "km_survival":round(sv,4),
                                 "ci_lower":round(lo,4),"ci_upper":round(hi,4),
                                 "logrank_p":p_str})
            print(f"  {tier:<22} {n:>5} {ev:>7}  " + "  ".join(f"{v:>18}" for v in vals))

km_df = pd.DataFrame(km_rows)
km_df.to_csv(f"{RESULTS_DIR}/results_km_summary.csv", index=False)
display(km_df)
print(f"\n  💾 results_km_summary: {len(km_df):,} rows")

# ── 2. Nelson-Aalen cumulative hazard ──────────────────────────────────────────
print("\n\n── 2. Nelson–Aalen cumulative hazard at checkpoints ──")
na_rows = []
for fig_stem, (df_in, classifiers) in FIG_GROUPS.items():
    for cls_label, tier_col in classifiers:
        if tier_col not in df_in.columns: continue
        sub = df_in.dropna(subset=[tier_col])
        print(f"\n  {fig_stem} | {cls_label}")
        print(f"  {'Tier':<22} {'N':>5}  " + "  ".join(f"{l:>18}" for l in TAU_LABELS))
        print("  " + "─"*100)
        for tier in TIER_ORDER:
            mask = sub[tier_col] == tier
            n    = mask.sum()
            if n < 5: continue
            naf  = NelsonAalenFitter()
            naf.fit(sub.loc[mask,"OS_TIME"], sub.loc[mask,"OS_EVENT"])
            vals = []
            for t in TAU_CHECKPOINTS:
                i  = _nearest(naf.cumulative_hazard_.index, t)
                ch = naf.cumulative_hazard_.iloc[i,0]
                lo = naf.confidence_interval_.iloc[i,0]
                hi = naf.confidence_interval_.iloc[i,1]
                vals.append(f"{ch:.3f} [{lo:.3f}–{hi:.3f}]")
                na_rows.append({"figure":fig_stem,"classifier":cls_label,"tier":tier,
                                 "n":n,"timepoint":t,
                                 "timepoint_label":TAU_LABELS[TAU_CHECKPOINTS.index(t)],
                                 "cumulative_hazard":round(ch,4),
                                 "ci_lower":round(lo,4),"ci_upper":round(hi,4)})
            print(f"  {tier:<22} {n:>5}  " + "  ".join(f"{v:>18}" for v in vals))

na_df = pd.DataFrame(na_rows)
na_df.to_csv(f"{RESULTS_DIR}/results_na_summary.csv", index=False)
display(na_df)
print(f"\n  💾 results_na_summary: {len(na_df):,} rows")

# ── 3. CIF (Aalen-Johansen) ────────────────────────────────────────────────────
print("\n\n── 3. Cumulative incidence (Aalen–Johansen) at checkpoints ──")
cif_rows = []
for fig_stem, (df_in, classifiers) in FIG_GROUPS.items():
    for cls_label, tier_col in classifiers:
        if tier_col not in df_in.columns: continue
        sub_full = df_in.dropna(subset=[tier_col]).copy()
        for out_label, time_col, event_col in CIF_OUTCOMES:
            if time_col not in df_in.columns or event_col not in df_in.columns: continue
            sub = sub_full.dropna(subset=[time_col, event_col]).copy()
            sub["_AJ"] = make_aj_event(sub, event_col, "OS_EVENT")
            try:
                lr    = multivariate_logrank_test(sub[time_col], sub[tier_col],
                                                  (sub["_AJ"]==1).astype(int))
                p_val = lr.p_value
                p_str = "< 0.001" if p_val < 0.001 else f"{p_val:.3f}"
            except Exception:
                p_str = "—"
            print(f"\n  {fig_stem} | {cls_label} | {out_label}  (log-rank p {p_str})")
            print(f"  {'Tier':<22} {'N':>5} {'Events':>7} {'Comp.':>7}  " +
                  "  ".join(f"{l:>18}" for l in TAU_LABELS))
            print("  " + "─"*115)
            for tier in TIER_ORDER:
                mask = sub[tier_col] == tier
                n    = mask.sum()
                if n < 5: continue
                sub_t = sub[mask]
                n_ev  = int((sub_t["_AJ"]==1).sum())
                n_cd  = int((sub_t["_AJ"]==2).sum())
                if n_ev < 2:
                    print(f"  {tier:<22} {n:>5} {n_ev:>7} {n_cd:>7}  (insufficient events)")
                    continue
                try:
                    ajf = AalenJohansenFitter(calculate_variance=True)
                    ajf.fit(sub_t[time_col], sub_t["_AJ"], event_of_interest=1)
                    cif_s = ajf.cumulative_density_["CIF_1"]
                    var_s = ajf.variance_
                    vals  = []
                    for t in TAU_CHECKPOINTS:
                        i  = _nearest(cif_s.index, t)
                        cv = float(cif_s.iloc[i])
                        se = np.sqrt(max(float(var_s.iloc[i]), 0))
                        lo = max(cv - 1.96*se, 0); hi = min(cv + 1.96*se, 1)
                        vals.append(f"{cv:.3f} [{lo:.3f}–{hi:.3f}]")
                        cif_rows.append({"figure":fig_stem,"classifier":cls_label,
                                          "outcome":out_label,"tier":tier,
                                          "n":n,"n_events":n_ev,"n_competing":n_cd,
                                          "timepoint":t,
                                          "timepoint_label":TAU_LABELS[TAU_CHECKPOINTS.index(t)],
                                          "cif":round(cv,4),
                                          "ci_lower":round(lo,4),"ci_upper":round(hi,4),
                                          "logrank_p":p_str})
                    print(f"  {tier:<22} {n:>5} {n_ev:>7} {n_cd:>7}  " +
                          "  ".join(f"{v:>18}" for v in vals))
                except Exception as e:
                    print(f"  {tier:<22}  ⚠ AJ failed: {e}")

cif_df = pd.DataFrame(cif_rows)
cif_df.to_csv(f"{RESULTS_DIR}/results_cif_summary.csv", index=False)
display(cif_df)
print(f"\n  💾 results_cif_summary: {len(cif_df):,} rows")

# ── 4. SMD summary ─────────────────────────────────────────────────────────────
print("\n\n── 4. Love plot SMD values ──")
SMD_CAP  = 2.0
non_ref  = [t for t in TIER_ORDER if t != "Baseline"]
smd_rows = []

for label, smd_dict in smd_cache.items():
    cohort = "OW" if any(df_in is df_ow for df_in, _, lbl in CLASSIFIERS if lbl==label) else "CW"
    print(f"\n  {clean_label(label)}  ({cohort})")
    print(f"  {'Covariate':<28}  " + "  ".join(f"{t:>10}" for t in non_ref))
    print("  " + "─"*65)
    for cov in COVARIATE_ORDER:
        if cov not in smd_dict: continue
        vals = []
        for tier in non_ref:
            v = smd_dict[cov].get(tier, np.nan)
            if np.isnan(v):
                vals.append(f"{'—':>10}")
            else:
                flag = "▶" if v > SMD_CAP else ("*" if v > 0.10 else " ")
                vals.append(f"{min(v,SMD_CAP):>9.3f}{flag}")
            smd_rows.append({"classifier":clean_label(label),"cohort":cohort,
                              "covariate":cov,"tier_vs_baseline":tier,
                              "smd_raw":    round(float(v),4) if not np.isnan(v) else None,
                              "smd_capped": round(min(float(v),SMD_CAP),4) if not np.isnan(v) else None,
                              "exceeds_0.10": bool(v > 0.10) if not np.isnan(v) else False,
                              "exceeds_0.25": bool(v > 0.25) if not np.isnan(v) else False,
                              "was_capped":   bool(v > SMD_CAP) if not np.isnan(v) else False})
        print(f"  {cov:<28}  " + "  ".join(vals))

smd_df = pd.DataFrame(smd_rows)
smd_df.to_csv(f"{RESULTS_DIR}/results_smd_summary.csv", index=False)
display(smd_df)
print(f"\n  💾 results_smd_summary: {len(smd_df):,} rows")

print("\n" + "="*70)
print("✅ All results summary tables complete")
print("="*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 14 — H4: Evidence Transparency Analysis
# Inputs:  df_cw, TIER_TO_NUM  (from Cell 10)
# Outputs: display + /content/results/h4_*.csv
# ═══════════════════════════════════════════════════════════════════════════════
from IPython.display import display, HTML, Markdown
import re
from collections import defaultdict

# ── Evidence dimension patterns ────────────────────────────────────────────────
DIMENSION_PATTERNS = {
    "Biomarker values": [
        "hba1c","egfr","glucose","creatinine","uacr","albumin",
        "blood pressure","cholesterol","ldl","hdl","triglyceride",
        "haemoglobin","hemoglobin","lab","laboratory","level","value",
        "measurement","result","reading","test",
    ],
    "Medical history & complications": [
        "retinopathy","nephropathy","neuropathy","cvd","cardiovascular",
        "stroke","myocardial","infarction","amputation","blindness",
        "dialysis","transplant","ulcer","macular","edema","oedema",
        "complication","history","diagnosed","established","chronic",
        "ckd","renal","kidney","heart","failure","peripheral",
    ],
    "Treatment intensity": [
        "insulin","metformin","glp","sglt","dpp","sulfonylurea",
        "medication","drug","prescription","treatment","therapy",
        "dose","escalat","intensif","pharmacolog","agent","class",
        "polypharmacy","combination",
    ],
    "Imaging & procedures": [
        "imaging","scan","ultrasound","ecg","echocardiogram","xray",
        "x-ray","fundus","photograph","biopsy","procedure","surgery",
        "operation","ophthalmolog",
    ],
    "Sociodemographics": [
        "age","sex","gender","male","female","ethnicity","race",
        "socioeconomic","smoking","alcohol","bmi","obesity","weight",
        "lifestyle","occupation","education",
    ],
    "Encounter patterns": [
        "admission","hospitalisation","hospitalization","emergency",
        "encounter","visit","outpatient","inpatient","referral",
        "specialist","appointment","follow-up","followup","attendance",
    ],
}
DIMENSIONS = list(DIMENSION_PATTERNS.keys())

def detect_dimensions(text):
    if not isinstance(text, str) or len(text.strip()) == 0: return set()
    text_l = text.lower()
    return {dim for dim, kws in DIMENSION_PATTERNS.items() if any(kw in text_l for kw in kws)}

def tier_diff(tier_a, tier_b):
    return TIER_TO_NUM.get(tier_a, 0) - TIER_TO_NUM.get(tier_b, 0)

def direction_label(diff):
    if diff == 0: return "Concordant"
    if diff > 0:  return f"LLM ↑ {abs(diff)} tier{'s' if abs(diff)>1 else ''}"
    return             f"LLM ↓ {abs(diff)} tier{'s' if abs(diff)>1 else ''}"

# ── Build H4 working dataframe ─────────────────────────────────────────────────
needed_cols = ["PATIENT","LLM_FOUR_TIER","OW_FOUR_TIER",
               "LLM_ASSESSOR_A_TIER","LLM_ASSESSOR_B_TIER",
               "OW_ASSESSOR_A_TIER","OW_ASSESSOR_B_TIER",
               "LLM_KEY_EVIDENCE","OW_KEY_EVIDENCE",
               "LLM_ASSESSORS_AGREE","OW_ASSESSORS_AGREE",
               "t5_young_tier","t5_cooper_tier",
               "t5_young_evidence","t5_cooper_evidence",
               "AGE_AT_T5","GENDER"]
h4 = df_cw[[c for c in needed_cols if c in df_cw.columns]].copy()

# Agreement flags
h4["agree_young_cw"]  = h4["LLM_FOUR_TIER"] == h4["t5_young_tier"]
h4["agree_cooper_cw"] = h4["LLM_FOUR_TIER"] == h4["t5_cooper_tier"]
h4["agree_young_ow"]  = h4["OW_FOUR_TIER"]  == h4["t5_young_tier"]
h4["agree_cooper_ow"] = h4["OW_FOUR_TIER"]  == h4["t5_cooper_tier"]
h4["agree_cw_ow"]     = h4["LLM_FOUR_TIER"] == h4["OW_FOUR_TIER"]
h4["agree_gts"]       = h4["t5_young_tier"] == h4["t5_cooper_tier"]

# Disagreement direction
h4["diff_cw_young"]  = h4.apply(lambda r: tier_diff(r["LLM_FOUR_TIER"], r["t5_young_tier"]),  axis=1)
h4["diff_cw_cooper"] = h4.apply(lambda r: tier_diff(r["LLM_FOUR_TIER"], r["t5_cooper_tier"]), axis=1)
h4["diff_ow_young"]  = h4.apply(lambda r: tier_diff(r["OW_FOUR_TIER"],  r["t5_young_tier"]),  axis=1)
h4["diff_ow_cooper"] = h4.apply(lambda r: tier_diff(r["OW_FOUR_TIER"],  r["t5_cooper_tier"]), axis=1)

# Dimension detection
h4["dims_cw"]     = h4["LLM_KEY_EVIDENCE"].apply(detect_dimensions)
h4["dims_ow"]     = h4["OW_KEY_EVIDENCE"].apply(detect_dimensions)
h4["dims_young"]  = h4["t5_young_evidence"].apply(detect_dimensions) if "t5_young_evidence" in h4 else [set()]*len(h4)
h4["dims_cooper"] = h4["t5_cooper_evidence"].apply(detect_dimensions) if "t5_cooper_evidence" in h4 else [set()]*len(h4)

disagree_cw  = h4[~h4["agree_young_cw"] | ~h4["agree_cooper_cw"]].copy()
agree_all_cw = h4[ h4["agree_young_cw"] &  h4["agree_cooper_cw"]].copy()

print(f"Total CW patients:            {len(h4)}")
print(f"Agree with BOTH GTs:          {len(agree_all_cw)}")
print(f"Disagree with ≥1 GT:          {len(disagree_cw)}")
print(f"Disagree with BOTH GTs:       {len(h4[~h4['agree_young_cw'] & ~h4['agree_cooper_cw']])}")
print(f"GTs agree with each other:    {h4['agree_gts'].sum()}")

# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 1 — Evidence Dimension Frequency
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("COMPONENT 1 — Evidence Dimension Frequency")
print("="*70)

comp1_rows = []
for _, row in h4.iterrows():
    for dim in DIMENSIONS:
        in_cw    = dim in row["dims_cw"]
        in_ow    = dim in row["dims_ow"]
        in_young = dim in row["dims_young"]
        in_cooper= dim in row["dims_cooper"]
        comp1_rows.append({
            "patient":            row["PATIENT"],
            "dimension":          dim,
            "in_cw_evidence":     in_cw,
            "in_ow_evidence":     in_ow,
            "in_young_evidence":  in_young,
            "in_cooper_evidence": in_cooper,
            "cw_vs_young":  ("Concordant" if in_cw==in_young  else ("LLM only" if in_cw else "GT only")),
            "cw_vs_cooper": ("Concordant" if in_cw==in_cooper else ("LLM only" if in_cw else "GT only")),
            "cw_vs_ow":     ("Concordant" if in_cw==in_ow     else "Discordant"),
            "cw_young_disagree":  not row["agree_young_cw"],
            "cw_cooper_disagree": not row["agree_cooper_cw"],
        })

comp1_df = pd.DataFrame(comp1_rows)
disagree_patients = set(disagree_cw["PATIENT"])
comp1_disagree    = comp1_df[comp1_df["patient"].isin(disagree_patients)]

for gt_col, gt_lbl in [("cw_vs_young","Young GT"), ("cw_vs_cooper","Cooper GT")]:
    print(f"\nDimension presence in disagreeing cases — CW vs {gt_lbl}:")
    print(f"  {'Dimension':<35} {'Concordant':>11} {'LLM only':>9} {'GT only':>9} {'% discordant':>13}")
    print("  " + "─"*80)
    for dim in DIMENSIONS:
        sub   = comp1_disagree[comp1_disagree["dimension"]==dim]
        n_c   = (sub[gt_col]=="Concordant").sum()
        n_llm = (sub[gt_col]=="LLM only").sum()
        n_gt  = (sub[gt_col]=="GT only").sum()
        total = len(sub)
        pct   = (n_llm+n_gt)/total*100 if total > 0 else 0
        print(f"  {dim:<35} {n_c:>11} {n_llm:>9} {n_gt:>9} {pct:>12.1f}%")

comp1_df.to_csv(f"{RESULTS_DIR}/h4_dimension_coding.csv", index=False)
print(f"\n  💾 h4_dimension_coding: {len(comp1_df):,} rows")
display(comp1_df.head(20))

# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 2 — Per-patient comparison table
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("COMPONENT 2 — Per-patient Comparison (disagreement cases)")
print("="*70)

comp2_rows = []
for _, row in disagree_cw.iterrows():
    comp2_rows.append({
        "patient":              row["PATIENT"],
        "age":                  row.get("AGE_AT_T5","—"),
        "sex":                  row.get("GENDER","—"),
        "tier_young":           row["t5_young_tier"],
        "tier_cooper":          row["t5_cooper_tier"],
        "tier_cw_A":            row.get("LLM_ASSESSOR_A_TIER","—"),
        "tier_cw_B":            row.get("LLM_ASSESSOR_B_TIER","—"),
        "tier_cw_final":        row["LLM_FOUR_TIER"],
        "tier_ow_A":            row.get("OW_ASSESSOR_A_TIER","—"),
        "tier_ow_B":            row.get("OW_ASSESSOR_B_TIER","—"),
        "tier_ow_final":        row["OW_FOUR_TIER"],
        "cw_assessors_agree":   row["LLM_ASSESSORS_AGREE"],
        "ow_assessors_agree":   row["OW_ASSESSORS_AGREE"],
        "cw_ow_agree":          row["agree_cw_ow"],
        "gts_agree":            row["agree_gts"],
        "diff_cw_young":        row["diff_cw_young"],
        "diff_cw_cooper":       row["diff_cw_cooper"],
        "direction_cw_young":   direction_label(row["diff_cw_young"]),
        "direction_cw_cooper":  direction_label(row["diff_cw_cooper"]),
        "dims_cw":              ", ".join(sorted(row["dims_cw"]))    or "none detected",
        "dims_ow":              ", ".join(sorted(row["dims_ow"]))    or "none detected",
        "dims_young":           ", ".join(sorted(row["dims_young"])) or "none detected",
        "dims_cooper":          ", ".join(sorted(row["dims_cooper"]))or "none detected",
        "n_dims_cw":            len(row["dims_cw"]),
        "n_dims_ow":            len(row["dims_ow"]),
        "n_extra_dims_vs_young":len(row["dims_cw"] - row["dims_young"]),
        "cw_key_evidence":      row["LLM_KEY_EVIDENCE"],
        "ow_key_evidence":      row["OW_KEY_EVIDENCE"],
        "young_evidence":       row.get("t5_young_evidence",""),
        "cooper_evidence":      row.get("t5_cooper_evidence",""),
    })

comp2_df = pd.DataFrame(comp2_rows)
comp2_df.to_csv(f"{RESULTS_DIR}/h4_per_patient_comparison.csv", index=False)
print(f"  💾 h4_per_patient_comparison: {len(comp2_df):,} rows")
display(comp2_df[["patient","tier_young","tier_cooper","tier_cw_final",
                   "tier_ow_final","direction_cw_young","direction_cw_cooper",
                   "cw_assessors_agree","gts_agree"]].head(20))

# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 3 — Case selection
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("COMPONENT 3 — Illustrative Case Selection")
print("="*70)

# Top concordance
agree_all_cw["concordance_score"] = (
    agree_all_cw["agree_gts"].astype(int) * 3 +
    agree_all_cw["LLM_ASSESSORS_AGREE"].astype(int) * 2 +
    agree_all_cw["agree_cw_ow"].astype(int) * 2 +
    agree_all_cw["LLM_KEY_EVIDENCE"].str.len().fillna(0).clip(upper=500) / 100
)
top_concordance = agree_all_cw.nlargest(5, "concordance_score")

print("\n── Top 5 concordance candidates ──")
print(f"  {'Patient':<38} {'Young':>12} {'Cooper':>12} {'CW':>12} {'OW':>12} {'Score':>7}")
print("  " + "─"*98)
for _, r in top_concordance.iterrows():
    print(f"  {str(r['PATIENT']):<38} {str(r['t5_young_tier']):>12} "
          f"{str(r['t5_cooper_tier']):>12} {str(r['LLM_FOUR_TIER']):>12} "
          f"{str(r['OW_FOUR_TIER']):>12} {r['concordance_score']:>7.2f}")

# Top divergence
cw_disagree_both = h4[~h4["agree_young_cw"] & ~h4["agree_cooper_cw"]].copy()
cw_disagree_both["divergence_score"] = (
    cw_disagree_both["agree_gts"].astype(int) * 3 +
    cw_disagree_both["LLM_ASSESSORS_AGREE"].astype(int) * 2 +
    cw_disagree_both["agree_cw_ow"].astype(int) * 2 +
    ((cw_disagree_both["diff_cw_young"] * cw_disagree_both["diff_cw_cooper"] > 0)
     .astype(int) * 2) +
    cw_disagree_both["LLM_KEY_EVIDENCE"].str.len().fillna(0).clip(upper=1000) / 200
)
top_divergence = cw_disagree_both.nlargest(5, "divergence_score")

print("\n── Top 5 divergence candidates ──")
print(f"  {'Patient':<38} {'Young':>12} {'Cooper':>12} {'CW':>12} {'OW':>12} "
      f"{'ΔYng':>5} {'ΔCpr':>5} {'Score':>7}")
print("  " + "─"*105)
for _, r in top_divergence.iterrows():
    print(f"  {str(r['PATIENT']):<38} {str(r['t5_young_tier']):>12} "
          f"{str(r['t5_cooper_tier']):>12} {str(r['LLM_FOUR_TIER']):>12} "
          f"{str(r['OW_FOUR_TIER']):>12} {r['diff_cw_young']:>+5} "
          f"{r['diff_cw_cooper']:>+5} {r['divergence_score']:>7.2f}")

# Full evidence printout — top candidates
def print_case(r, label):
    print(f"\n{'─'*65}")
    print(f"{label}")
    print(f"  Patient:     {r['PATIENT']}")
    age = f"{r['AGE_AT_T5']:.0f}" if pd.notna(r.get('AGE_AT_T5')) else '—'
    print(f"  Age/Sex:     {age} / {r.get('GENDER','—')}")
    print(f"  Young GT:    {r['t5_young_tier']}")
    print(f"  Cooper GT:   {r['t5_cooper_tier']}")
    print(f"  CW final:    {r['LLM_FOUR_TIER']}  "
          f"(A={r.get('LLM_ASSESSOR_A_TIER','—')}, B={r.get('LLM_ASSESSOR_B_TIER','—')}, "
          f"agree={r['LLM_ASSESSORS_AGREE']})")
    print(f"  OW final:    {r['OW_FOUR_TIER']}  "
          f"(A={r.get('OW_ASSESSOR_A_TIER','—')}, B={r.get('OW_ASSESSOR_B_TIER','—')})")
    if pd.notna(r.get('t5_young_evidence')):
        print(f"\n  Young ev.:   {r['t5_young_evidence']}")
    if pd.notna(r.get('t5_cooper_evidence')):
        print(f"  Cooper ev.:  {r['t5_cooper_evidence']}")
    print(f"\n  CW evidence:\n    {r['LLM_KEY_EVIDENCE']}")
    print(f"\n  OW evidence:\n    {r['OW_KEY_EVIDENCE']}")

print("\n" + "="*70)
print("FULL EVIDENCE — Top concordance case")
print_case(top_concordance.iloc[0], "★ Best concordance case")

print("\n" + "="*70)
print("FULL EVIDENCE — Top 3 divergence cases")
for rank, (_, r) in enumerate(top_divergence.head(3).iterrows(), 1):
    print_case(r, f"Divergence rank {rank}  "
               f"(Δ Young={r['diff_cw_young']:+d}, Δ Cooper={r['diff_cw_cooper']:+d})")

# Save component 3
comp3_cols_conc = ["PATIENT","t5_young_tier","t5_cooper_tier","LLM_FOUR_TIER",
                   "OW_FOUR_TIER","LLM_ASSESSORS_AGREE","concordance_score",
                   "LLM_KEY_EVIDENCE","OW_KEY_EVIDENCE"]
comp3_cols_div  = ["PATIENT","t5_young_tier","t5_cooper_tier","LLM_FOUR_TIER",
                   "OW_FOUR_TIER","diff_cw_young","diff_cw_cooper",
                   "LLM_ASSESSORS_AGREE","agree_gts","agree_cw_ow","divergence_score",
                   "LLM_KEY_EVIDENCE","OW_KEY_EVIDENCE"]

top_concordance[[c for c in comp3_cols_conc if c in top_concordance.columns]].to_csv(
    f"{RESULTS_DIR}/h4_case_concordance_candidates.csv", index=False)
top_divergence[[c for c in comp3_cols_div if c in top_divergence.columns]].to_csv(
    f"{RESULTS_DIR}/h4_case_divergence_candidates.csv", index=False)
print(f"\n  💾 h4_case_concordance_candidates + h4_case_divergence_candidates saved")
print("\n✅ H4 Components 1–3 complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 16 — Illustrated Case Studies: Top 5 Concordance + Top 5 Divergence
# Inputs:  top_concordance, top_divergence, h4  (from Cell 14)
# Outputs: display + /content/results/h4_case_studies.csv
# ═══════════════════════════════════════════════════════════════════════════════
from IPython.display import display, HTML

# ── Tier badge colours ─────────────────────────────────────────────────────────
TIER_BADGE = {
    "Baseline":  ("🔵","#E3F2FD","#1565C0"),
    "Mild":      ("🟢","#E8F5E9","#2E7D32"),
    "Moderate":  ("🟠","#FFF3E0","#E65100"),
    "Critical":  ("🔴","#FFEBEE","#B71C1C"),
}

def badge(tier):
    if not isinstance(tier, str): return f'<span style="color:#999">—</span>'
    tier_clean = tier.strip()
    # normalise long names
    for short in ["Baseline","Mild","Moderate","Critical"]:
        if short.lower() in tier_clean.lower():
            tier_clean = short; break
    ico, bg, fg = TIER_BADGE.get(tier_clean, ("⚪","#F5F5F5","#333"))
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:4px;font-weight:600;font-size:12px;">'
            f'{ico} {tier_clean}</span>')

def agree_icon(val):
    if val is True  or val == 1: return '✅'
    if val is False or val == 0: return '❌'
    return '—'

# ── Auto-generate divergence interpretation ────────────────────────────────────
def interpret_divergence(row):
    ev     = str(row.get("LLM_KEY_EVIDENCE","")).lower()
    d_yng  = row.get("diff_cw_young", 0)
    d_cpr  = row.get("diff_cw_cooper", 0)
    gt_yng = str(row.get("t5_young_tier",""))
    gt_cpr = str(row.get("t5_cooper_tier",""))
    cw     = str(row.get("LLM_FOUR_TIER",""))

    reasons = []

    # Up-tiering patterns
    if d_yng > 0 or d_cpr > 0:
        if "insulin" in ev and ("pharmacotherapy" in ev or "proxy" in ev or "beta-cell" in ev):
            reasons.append("🔬 <b>Insulin-as-proxy trigger:</b> LLM applies a pharmacotherapy "
                           "proxy criterion — sustained insulin use signals beta-cell failure "
                           "even without documented complications. Neither Young DCSI nor Cooper "
                           "score this directly, so both GTs remain at Baseline.")
        if "coronary" in ev or "cardiovascular" in ev or "cvd" in ev or "ischemic" in ev:
            reasons.append("❤️ <b>Macrovascular blind spot:</b> LLM identifies cardiovascular "
                           "disease (CVD/CHD) as a standalone Tier 3 trigger. Young GT requires "
                           "diabetes-specific microvascular complications for severity scoring; "
                           "Cooper GT counts complication instances but CVD without microvascular "
                           "damage scores 0. Both GTs are algorithmically blind to macrovascular "
                           "severity in the absence of retinopathy/nephropathy/neuropathy.")
        if "rule 0a" in ev or "concurrent tier 3" in ev:
            reasons.append("⚡ <b>Rule 0a multi-domain escalation:</b> LLM identifies ≥2 "
                           "concurrent Tier 3 domains triggering mandatory escalation. "
                           "GT scoring systems use additive severity, not multiplicative "
                           "threshold rules, so the same clinical picture scores differently.")
        if "diagnostic uncertainty" in ev or "atypical" in ev or "classification" in ev:
            reasons.append("⚠️ <b>Diagnostic uncertainty flagged:</b> LLM raises concern about "
                           "T2D classification validity (e.g. insulin from diagnosis without "
                           "typical T2D progression pattern). GTs classify mechanically from "
                           "coded diagnoses without this contextual reasoning.")

    # Down-tiering patterns
    if d_yng < 0 or d_cpr < 0:
        reasons.append("⬇️ <b>Conservative down-tiering:</b> LLM assigns a lower tier than "
                       "one or both GTs. This may reflect LLM weighting of missing data, "
                       "uncertainty about complication severity, or a more holistic clinical "
                       "gestalt that discounts borderline GT criteria.")
        if "missing" in ev or "unavailable" in ev or "no documented" in ev:
            reasons.append("📭 <b>Missing data acknowledgement:</b> LLM explicitly flags "
                           "absence of key clinical data and adjusts severity downward, "
                           "whereas GT algorithms assign severity based on coded fields "
                           "regardless of overall data completeness.")

    # Pipeline divergence (CW ≠ OW)
    if not row.get("agree_cw_ow", True):
        reasons.append("🔀 <b>Inter-pipeline disagreement:</b> CW and OW pipelines reach "
                       "different conclusions, suggesting the clinical picture is ambiguous "
                       "and the divergence from GTs may not be a single systematic bias.")

    if not reasons:
        reasons.append("ℹ️ <b>Pattern unclear:</b> Evidence text does not match known "
                       "divergence templates. Manual review recommended.")

    return reasons

# ── HTML card builder ──────────────────────────────────────────────────────────
def render_case_card(row, rank, case_type):
    age    = f"{row['AGE_AT_T5']:.0f}" if pd.notna(row.get('AGE_AT_T5')) else '—'
    sex    = str(row.get('GENDER','—'))
    pid    = str(row['PATIENT'])[:8] + "…"

    header_color = "#1B5E20" if case_type == "concordance" else "#B71C1C"
    header_label = f"★ Concordance Case #{rank}" if case_type == "concordance" \
                   else f"✦ Divergence Case #{rank}"

    d_yng  = row.get("diff_cw_young",  0)
    d_cpr  = row.get("diff_cw_cooper", 0)
    delta_str = (f'<span style="color:#B71C1C;font-weight:600">'
                 f'Δ Young = {d_yng:+d} &nbsp;|&nbsp; Δ Cooper = {d_cpr:+d}</span>'
                 if case_type=="divergence" else
                 '<span style="color:#1B5E20;font-weight:600">All systems agree ✅</span>')

    reasons_html = ""
    if case_type == "divergence":
        reasons = interpret_divergence(row)
        reasons_html = "".join(
            f'<div style="margin:6px 0;padding:8px 12px;background:#FFF8E1;'
            f'border-left:3px solid #F9A825;border-radius:4px;font-size:12px;">'
            f'{r}</div>' for r in reasons)

    ev_cw    = str(row.get("LLM_KEY_EVIDENCE","—"))
    ev_ow    = str(row.get("OW_KEY_EVIDENCE","—"))
    ev_young = str(row.get("t5_young_evidence","—"))
    ev_coop  = str(row.get("t5_cooper_evidence","—"))

    assessor_row = (
        f'<tr><td style="padding:4px 8px;color:#666">CW assessors</td>'
        f'<td style="padding:4px 8px">{badge(row.get("LLM_ASSESSOR_A_TIER","—"))} '
        f'(A) &nbsp; {badge(row.get("LLM_ASSESSOR_B_TIER","—"))} (B) &nbsp; '
        f'{agree_icon(row.get("LLM_ASSESSORS_AGREE"))} agree</td></tr>'
        f'<tr><td style="padding:4px 8px;color:#666">OW assessors</td>'
        f'<td style="padding:4px 8px">{badge(row.get("OW_ASSESSOR_A_TIER","—"))} '
        f'(A) &nbsp; {badge(row.get("OW_ASSESSOR_B_TIER","—"))} (B) &nbsp; '
        f'{agree_icon(row.get("OW_ASSESSORS_AGREE"))} agree</td></tr>'
    )

    return f"""
    <div style="border:1px solid #DDD;border-radius:8px;margin:20px 0;
                font-family:Georgia,serif;overflow:hidden;box-shadow:0 2px 6px rgba(0,0,0,.08);">

      <!-- Header -->
      <div style="background:{header_color};color:white;padding:10px 16px;
                  display:flex;justify-content:space-between;align-items:center;">
        <span style="font-size:14px;font-weight:bold;">{header_label}</span>
        <span style="font-size:12px;opacity:.85;">Patient {pid} &nbsp;·&nbsp;
          Age {age} &nbsp;·&nbsp; Sex {sex}</span>
      </div>

      <div style="padding:14px 18px;">

        <!-- Tier table -->
        <table style="border-collapse:collapse;width:100%;margin-bottom:12px;font-size:13px;">
          <tr style="border-bottom:1px solid #EEE;">
            <td style="padding:5px 8px;color:#666;width:140px;">Young GT</td>
            <td style="padding:5px 8px;">{badge(row.get('t5_young_tier','—'))}</td>
            <td style="padding:5px 8px;color:#666;width:140px;">Cooper GT</td>
            <td style="padding:5px 8px;">{badge(row.get('t5_cooper_tier','—'))}</td>
          </tr>
          <tr style="border-bottom:1px solid #EEE;">
            <td style="padding:5px 8px;color:#666;">CW (consolidated)</td>
            <td style="padding:5px 8px;">{badge(row.get('LLM_FOUR_TIER','—'))}</td>
            <td style="padding:5px 8px;color:#666;">OW (consolidated)</td>
            <td style="padding:5px 8px;">{badge(row.get('OW_FOUR_TIER','—'))}</td>
          </tr>
          {assessor_row}
          <tr>
            <td style="padding:5px 8px;color:#666;">GTs agree</td>
            <td style="padding:5px 8px;">{agree_icon(row.get('agree_gts'))}</td>
            <td style="padding:5px 8px;color:#666;">CW = OW</td>
            <td style="padding:5px 8px;">{agree_icon(row.get('agree_cw_ow'))}</td>
          </tr>
        </table>

        <!-- Delta line -->
        <div style="margin:8px 0 12px;font-size:13px;">{delta_str}</div>

        <!-- Divergence interpretation -->
        {reasons_html}

        <!-- Evidence grid -->
        <table style="border-collapse:collapse;width:100%;margin-top:12px;font-size:11.5px;">
          <tr style="background:#F5F5F5;">
            <th style="padding:6px 10px;text-align:left;width:50%;border:1px solid #DDD;">
              Young GT evidence</th>
            <th style="padding:6px 10px;text-align:left;width:50%;border:1px solid #DDD;">
              Cooper GT evidence</th>
          </tr>
          <tr>
            <td style="padding:8px 10px;border:1px solid #EEE;vertical-align:top;
                        color:#444;line-height:1.5;">{ev_young}</td>
            <td style="padding:8px 10px;border:1px solid #EEE;vertical-align:top;
                        color:#444;line-height:1.5;">{ev_coop}</td>
          </tr>
          <tr style="background:#F5F5F5;">
            <th style="padding:6px 10px;text-align:left;border:1px solid #DDD;">
              CW key evidence</th>
            <th style="padding:6px 10px;text-align:left;border:1px solid #DDD;">
              OW key evidence</th>
          </tr>
          <tr>
            <td style="padding:8px 10px;border:1px solid #EEE;vertical-align:top;
                        line-height:1.5;">{ev_cw}</td>
            <td style="padding:8px 10px;border:1px solid #EEE;vertical-align:top;
                        line-height:1.5;">{ev_ow}</td>
          </tr>
        </table>

      </div>
    </div>"""

# ── Render all cases ───────────────────────────────────────────────────────────
display(Markdown("## Case Studies — Top 5 Concordance Cases"))
html_blocks = []
for rank, (_, row) in enumerate(top_concordance.head(5).iterrows(), 1):
    h = render_case_card(row, rank, "concordance")
    html_blocks.append(h)
    display(HTML(h))

display(Markdown("## Case Studies — Top 5 Divergence Cases"))
for rank, (_, row) in enumerate(top_divergence.head(5).iterrows(), 1):
    h = render_case_card(row, rank, "divergence")
    html_blocks.append(h)
    display(HTML(h))

# ── Save flat CSV of all 10 cases ─────────────────────────────────────────────
save_cols = ["PATIENT","AGE_AT_T5","GENDER",
             "t5_young_tier","t5_cooper_tier",
             "LLM_FOUR_TIER","OW_FOUR_TIER",
             "LLM_ASSESSOR_A_TIER","LLM_ASSESSOR_B_TIER",
             "OW_ASSESSOR_A_TIER","OW_ASSESSOR_B_TIER",
             "LLM_ASSESSORS_AGREE","OW_ASSESSORS_AGREE",
             "agree_gts","agree_cw_ow",
             "t5_young_evidence","t5_cooper_evidence",
             "LLM_KEY_EVIDENCE","OW_KEY_EVIDENCE"]

conc_out = top_concordance.head(5)[[c for c in save_cols if c in top_concordance.columns]].copy()
conc_out.insert(0, "case_type", "concordance")
conc_out.insert(1, "rank",      range(1,6))
conc_out["diff_cw_young"]  = 0
conc_out["diff_cw_cooper"] = 0

div_out  = top_divergence.head(5)[[c for c in save_cols if c in top_divergence.columns]].copy()
div_out.insert(0, "case_type", "divergence")
div_out.insert(1, "rank",      range(1,6))
div_out["diff_cw_young"]  = top_divergence.head(5)["diff_cw_young"].values
div_out["diff_cw_cooper"] = top_divergence.head(5)["diff_cw_cooper"].values

case_studies_df = pd.concat([conc_out, div_out], ignore_index=True)
case_studies_df.to_csv(f"{RESULTS_DIR}/h4_case_studies.csv", index=False)
print(f"\n💾 h4_case_studies: {len(case_studies_df)} cases → {RESULTS_DIR}/h4_case_studies.csv")
print("✅ Case studies complete")